# NOx Dispersion Prediction — 3D UNet (Physics-Informed)
#### Dataset `dataset_subset_3000.h5` · 3000 scénarios · H100

**Pipeline :**
1. Chargement HDF5 streaming (3000 scénarios, grilles 140×140 / 300×300 / 500×500)
2. 3D UNet : 7 canaux d'entrée (vent + géométrie + émissions) → concentration NOx
3. Loss composite PINN : MSE pondérée near-wall + conservation de masse + TV
4. Reprise automatique sur crash · log CSV/JSON par époque
5. Visualisations complètes : courbes, tranches 2D, isosurfaces 3D, vidéo


In [1]:
# -- Cellule 0b : Verify PyTorch environment --------------------
import sys

print("This notebook no longer uninstalls or reinstalls torch automatically.")
print("Python:", sys.version.split()[0])

try:
    import torch

    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print("VRAM (GB):", round(total_gb, 1))
except Exception as exc:
    print("Torch import failed:", exc)
    print("Install a working PyTorch build on the server before training.")



This notebook no longer uninstalls or reinstalls torch automatically.
Python: 3.13.5
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA H100 PCIe
VRAM (GB): 84.9


In [2]:
# -- Cellule 1 : Dependencies --------------------------------------
import importlib

REQUIRED_PACKAGES = [
    "h5py",
    "matplotlib",
    "seaborn",
    "tqdm",
    "scipy",
    "sklearn",
    "plotly",
    "imageio",
]

missing = [
    name for name in REQUIRED_PACKAGES
    if importlib.util.find_spec(name) is None
]

if missing:
    raise ModuleNotFoundError(
        "Missing packages: "
        + ", ".join(missing)
        + "\nInstall them once on the server, for example:\n"
        + "pip install h5py matplotlib seaborn tqdm scipy scikit-learn "
          "plotly imageio[ffmpeg]"
    )

print("All notebook dependencies are available.")


All notebook dependencies are available.


In [3]:
# -- Cellule 2 : Imports and GPU setup ---------------------------------
import os
import sys
import json
import csv
import time
import shutil
import random
import math
import zipfile
import importlib.util
import platform
import hashlib
import traceback
import gc
from pathlib import Path
from collections import Counter, defaultdict
from contextlib import nullcontext

import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
sns.set_style("whitegrid")

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BF16_OK = bool(
    DEVICE == "cuda"
    and hasattr(torch.cuda, "is_bf16_supported")
    and torch.cuda.is_bf16_supported()
)

if DEVICE == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)")
    torch.backends.cudnn.benchmark = True
else:
    gpu_name = "cpu"
    gpu_mem = 0.0
    print("No CUDA GPU detected. The notebook can still run, but training will be slow.")

print(f"PyTorch {torch.__version__} | Device: {DEVICE} | BF16: {BF16_OK}")



GPU: NVIDIA H100 PCIe (84.9 GB VRAM)
PyTorch 2.11.0+cu128 | Device: cuda | BF16: True


In [4]:
# -- Cellule 3 : Configuration and path detection ----------------
_FILENAME = "dataset_subset_3000.h5"


def as_jsonable(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


def safe_json_dump(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    tmp_path.write_text(json.dumps(data, indent=2), encoding="utf-8")
    tmp_path.replace(path)


def append_event(event, **payload):
    row = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
        "event": event,
    }
    row.update({key: as_jsonable(value) for key, value in payload.items()})
    event_path = WORK_DIR / "logs" / "events.jsonl"
    with open(event_path, "a", encoding="utf-8") as handle:
        handle.write(json.dumps(row) + "\n")


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def safe_exists(path):
    try:
        return Path(path).exists()
    except (OSError, PermissionError):
        return False


DIRECT_CANDIDATES = [
    Path("/workspace/data") / _FILENAME,
    Path("/workspace") / _FILENAME,
    Path.cwd() / _FILENAME,
    Path.cwd() / "data" / _FILENAME,
    Path.cwd().parent / _FILENAME,
]


def find_dataset_path(filename):
    env_candidates = [
        os.environ.get("NOX_H5_PATH"),
        os.environ.get("DATASET_H5_PATH"),
    ]
    for raw in env_candidates:
        if raw and safe_exists(raw):
            return Path(raw).resolve()

    for candidate in DIRECT_CANDIDATES:
        if safe_exists(candidate):
            return candidate.resolve()

    raise FileNotFoundError(
        "dataset_subset_3000.h5 was not found.\n"
        "Upload it manually to /workspace/data/dataset_subset_3000.h5 on the server,\n"
        "or set NOX_H5_PATH to the exact file location."
    )


H5_PATH = find_dataset_path(_FILENAME)
WORK_DIR = Path("./nox_dispersion_run")
GEOM_MODE = "sdf"

CONFIG = {
    "h5_path": str(H5_PATH),
    "work_dir": str(WORK_DIR),
    "geom_mode": GEOM_MODE,
    "base_ch": 32,
    "crop_yx": 224,
    "batch_size": 8,
    "num_workers": min(12, os.cpu_count() or 4),
    "n_epochs": 200,
    "lr": 3e-4,
    "lr_min": 1e-6,
    "weight_decay": 1e-5,
    "use_amp": bool(DEVICE == "cuda"),
    "amp_dtype": "bfloat16" if BF16_OK else "float16",
    "lambda_mass": 0.1,
    "lambda_tv": 0.01,
    "lambda_plume_focus": 1.0,
    "lambda_active_mae": 0.15,
    "boundary_band": 0.15,
    "boundary_weight": 5.0,
    "background_voxel_weight": 0.05,
    "active_voxel_weight": 24.0,
    "high_voxel_weight": 96.0,
    "plume_active_threshold_ugm3": 10.0,
    "plume_high_threshold_ugm3": 50.0,
    "val_split": 0.10,
    "test_split": 0.10,
    "save_every": 5,
    "seed": 42,
    "resume_training": True,
    "reuse_existing_split": False,
    "exclude_zero_case_train": True,
    "train_crop_mode": "hybrid_focus",
    "val_crop_mode": "seeded_target",
    "focus_top_quantile": 0.999,
    "train_target_focus_prob": 0.85,
    "train_emission_focus_prob": 0.10,
    "train_random_crop_prob": 0.05,
    "selection_eval_cases": 16,
    "classification_bins": [0, 10, 50, 100, 200, 500],
    "classification_labels": [
        "0-10",
        "10-50",
        "50-100",
        "100-200",
        "200-500",
        ">500",
    ],
    "selection_metric": "selection_plume_rmse",
    "selection_mode": "min",
    "val_metric_points_per_batch": 12000,
    "full_eval_tile_yx": 224,
    "full_eval_overlap": 56,
    "notes": (
        "Accuracy-first H100 configuration with larger context, corrected wind-aware "
        "augmentation, plume-focused weighting, and full-volume validation subset selection."
    ),
}

for subdir in ["checkpoints", "logs", "outputs", "visualizations"]:
    (WORK_DIR / subdir).mkdir(parents=True, exist_ok=True)

set_global_seed(CONFIG["seed"])
safe_json_dump(CONFIG, WORK_DIR / "config.json")

dataset_size_gb = H5_PATH.stat().st_size / 1e9
SYSTEM_INFO = {
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
    "cwd": str(Path.cwd()),
    "python": sys.version,
    "platform": platform.platform(),
    "pytorch": torch.__version__,
    "device": DEVICE,
    "gpu_name": gpu_name,
    "gpu_memory_gb": round(gpu_mem, 3),
    "bf16_supported": BF16_OK,
    "dataset_path": str(H5_PATH),
    "dataset_size_gb": round(dataset_size_gb, 3),
}
safe_json_dump(SYSTEM_INFO, WORK_DIR / "logs" / "system_info.json")

append_event(
    "run_initialized",
    dataset_path=str(H5_PATH),
    dataset_size_gb=round(dataset_size_gb, 3),
    device=DEVICE,
    amp_dtype=CONFIG["amp_dtype"],
    selection_metric=CONFIG["selection_metric"],
    train_crop_mode=CONFIG["train_crop_mode"],
)

print(f"WORK_DIR : {WORK_DIR.resolve()}")
print(f"H5_PATH  : {H5_PATH}")
print(f"Dataset  : {dataset_size_gb:.2f} GB")
print(
    f"Geom={CONFIG['geom_mode']} | Crop={CONFIG['crop_yx']} | "
    f"Batch={CONFIG['batch_size']} | AMP={CONFIG['use_amp']} ({CONFIG['amp_dtype']})"
)
print(
    f"Train crop mode={CONFIG['train_crop_mode']} | "
    f"Val crop mode={CONFIG['val_crop_mode']} | "
    f"Selection metric={CONFIG['selection_metric']}"
)


WORK_DIR : /workspace/nox_dispersion_run
H5_PATH  : /workspace/data/dataset_subset_3000.h5
Dataset  : 124.60 GB
Geom=sdf | Crop=224 | Batch=8 | AMP=True (bfloat16)
Train crop mode=hybrid_focus | Val crop mode=seeded_target | Selection metric=selection_plume_rmse


In [5]:
# -- Cellule 3b : Manual upload note -----------------------------------
print("Manual upload mode is active.")
print("No Google Drive or remote fetch is performed by this notebook.")
print(f"Expected dataset path on the server: {CONFIG['h5_path']}")

losses_candidates = [
    Path("provided/losses.py"),
    Path("../provided/losses.py"),
]
losses_path = next((path for path in losses_candidates if path.exists()), None)
if losses_path is None:
    print("Reminder: provided/losses.py must be reachable from the notebook directory.")
else:
    print(f"losses.py found at: {losses_path.resolve()}")



Manual upload mode is active.
No Google Drive or remote fetch is performed by this notebook.
Expected dataset path on the server: /workspace/data/dataset_subset_3000.h5
losses.py found at: /workspace/provided/losses.py


In [6]:
# -- Cellule 4 : Dataset exploration and metadata logging ---------------
RESUME_CANDIDATES = [
    Path("provided/dataset_subset_3000_resume.json"),
    Path("../provided/dataset_subset_3000_resume.json"),
]


def load_dataset_resume():
    resume_path = next((path for path in RESUME_CANDIDATES if path.exists()), None)
    if resume_path is None:
        return None
    return json.loads(resume_path.read_text(encoding="utf-8"))


DATASET_RESUME = load_dataset_resume()


def collect_scenario_metadata(h5_path):
    with h5py.File(h5_path, "r") as handle:
        all_keys = sorted(handle.keys())
        scenario_meta = {}
        category_counter = Counter()
        stability_counter = Counter()
        nx_counter = Counter()
        zero_case_proxy_count = 0
        examples = []

        for idx, key in enumerate(tqdm(all_keys, desc="Read scenario metadata")):
            group = handle[key]
            attrs = {
                name: as_jsonable(group.attrs[name])
                for name in group.attrs.keys()
            }
            shape = tuple(int(v) for v in group["concentration_grid"].shape)

            row = {
                "category": int(attrs.get("category", -1)),
                "stability_class": str(attrs.get("stability_class", "?")),
                "Nx": int(attrs.get("Nx", shape[-1])),
                "Ny": int(attrs.get("Ny", shape[-2])),
                "Nz": int(shape[0]),
                "shape": list(shape),
                "U_10": float(attrs.get("U_10", 0.0)),
                "n_active_particles": int(attrs.get("n_active_particles", 0)),
                "n_buildings": int(attrs.get("n_buildings", 0)),
                "zero_case_proxy": int(attrs.get("n_active_particles", 0)) == 0,
            }
            scenario_meta[key] = row

            category_counter[row["category"]] += 1
            stability_counter[row["stability_class"]] += 1
            nx_counter[row["Nx"]] += 1
            zero_case_proxy_count += int(row["zero_case_proxy"])

            if idx < 5:
                examples.append({"key": key, **row})

    summary = {
        "total_scenarios": len(all_keys),
        "category_distribution": dict(sorted(category_counter.items())),
        "stability_distribution": dict(sorted(stability_counter.items())),
        "nx_distribution": dict(sorted(nx_counter.items())),
        "zero_case_proxy_count": zero_case_proxy_count,
        "zero_case_proxy_ratio": round(zero_case_proxy_count / max(len(all_keys), 1), 4),
        "examples": examples,
    }

    if DATASET_RESUME is not None:
        summary["dataset_resume_meta"] = DATASET_RESUME.get("meta", {})
        summary["dataset_resume_distribution"] = DATASET_RESUME.get("distribution_scenarios", {})
        summary["dataset_resume_quality"] = DATASET_RESUME.get("qualite_donnees", {})

    return all_keys, scenario_meta, summary


ALL_KEYS, ALL_META, DATASET_INSPECTION = collect_scenario_metadata(CONFIG["h5_path"])
safe_json_dump(DATASET_INSPECTION, WORK_DIR / "logs" / "dataset_inspection.json")
safe_json_dump(ALL_META, WORK_DIR / "logs" / "scenario_metadata.json")

append_event(
    "dataset_inspected",
    total_scenarios=len(ALL_KEYS),
    zero_case_proxy_count=DATASET_INSPECTION["zero_case_proxy_count"],
)

print(f"Total scenarios: {len(ALL_KEYS)}")
print("Categories :", DATASET_INSPECTION["category_distribution"])
print("Stability  :", DATASET_INSPECTION["stability_distribution"])
print("Grid sizes :", DATASET_INSPECTION["nx_distribution"])
print(
    "Zero-case proxy:",
    DATASET_INSPECTION["zero_case_proxy_count"],
    f"({DATASET_INSPECTION['zero_case_proxy_ratio'] * 100:.2f}%)",
)



Read scenario metadata: 100%|██████████| 3000/3000 [00:02<00:00, 1245.48it/s]


Total scenarios: 3000
Categories : {1: 750, 2: 750, 3: 750, 4: 750}
Stability  : {'I': 500, 'II': 500, 'III/1': 500, 'III/2': 500, 'IV': 500, 'V': 500}
Grid sizes : {140: 1500, 300: 750, 500: 750}
Zero-case proxy: 422 (14.07%)


In [7]:
# -- Cellule 5 : Stratified train / val / test split --------------------
SPLIT_PATH = WORK_DIR / "data_split.json"


def try_make_split(keys, labels, test_fraction, val_fraction, seed):
    temp_fraction = test_fraction + val_fraction
    if labels is None:
        train_keys, temp_keys = train_test_split(
            keys,
            test_size=temp_fraction,
            random_state=seed,
            shuffle=True,
        )
        val_keys, test_keys = train_test_split(
            temp_keys,
            test_size=test_fraction / temp_fraction,
            random_state=seed,
            shuffle=True,
        )
        return sorted(train_keys), sorted(val_keys), sorted(test_keys)

    train_keys, temp_keys, _, temp_labels = train_test_split(
        keys,
        labels,
        test_size=temp_fraction,
        random_state=seed,
        shuffle=True,
        stratify=labels,
    )
    val_keys, test_keys = train_test_split(
        temp_keys,
        test_size=test_fraction / temp_fraction,
        random_state=seed,
        shuffle=True,
        stratify=temp_labels,
    )
    return sorted(train_keys), sorted(val_keys), sorted(test_keys)


def summarize_split(keys):
    category_counter = Counter()
    stability_counter = Counter()
    nx_counter = Counter()
    zero_case_proxy_count = 0
    u10_values = []

    for key in keys:
        meta = ALL_META[key]
        category_counter[meta["category"]] += 1
        stability_counter[meta["stability_class"]] += 1
        nx_counter[meta["Nx"]] += 1
        zero_case_proxy_count += int(meta["zero_case_proxy"])
        u10_values.append(meta["U_10"])

    return {
        "n": len(keys),
        "category_distribution": dict(sorted(category_counter.items())),
        "stability_distribution": dict(sorted(stability_counter.items())),
        "nx_distribution": dict(sorted(nx_counter.items())),
        "zero_case_proxy_count": zero_case_proxy_count,
        "zero_case_proxy_ratio": round(zero_case_proxy_count / max(len(keys), 1), 4),
        "u10_mean": round(float(np.mean(u10_values)), 4) if u10_values else None,
    }


keys_sorted = sorted(ALL_KEYS)
strat_fine = [f"{ALL_META[key]['category']}|{ALL_META[key]['stability_class']}" for key in keys_sorted]
strat_category = [str(ALL_META[key]["category"]) for key in keys_sorted]

strategy_order = [
    ("category_and_stability", strat_fine),
    ("category_only", strat_category),
    ("random_fallback", None),
]

split_error = None
for strategy_name, strategy_labels in strategy_order:
    try:
        train_keys_raw, val_keys_raw, test_keys_raw = try_make_split(
            keys_sorted,
            strategy_labels,
            CONFIG["test_split"],
            CONFIG["val_split"],
            CONFIG["seed"],
        )
        SPLIT_STRATEGY = strategy_name
        split_error = None
        break
    except ValueError as exc:
        split_error = str(exc)
else:
    raise RuntimeError(f"Could not build a split: {split_error}")

removed_zero_case_train = []
if CONFIG.get("exclude_zero_case_train", False):
    removed_zero_case_train = [key for key in train_keys_raw if ALL_META[key]["zero_case_proxy"]]
    TRAIN_KEYS = sorted([key for key in train_keys_raw if not ALL_META[key]["zero_case_proxy"]])
else:
    TRAIN_KEYS = sorted(train_keys_raw)

VAL_KEYS = sorted(val_keys_raw)
TEST_KEYS = sorted(test_keys_raw)


def stable_selection_hash(text):
    return int(hashlib.sha256(text.encode("utf-8")).hexdigest()[:16], 16)


def select_validation_cases(keys, n_cases):
    non_zero_keys = [key for key in keys if not ALL_META[key]["zero_case_proxy"]]
    pool = non_zero_keys or list(keys)
    ordered = sorted(pool, key=lambda key: stable_selection_hash(f"val-selection:{key}"))
    return ordered[: min(int(n_cases), len(ordered))]


VAL_SELECTION_KEYS = select_validation_cases(VAL_KEYS, CONFIG.get("selection_eval_cases", 16))
TRAIN_SIGNATURE = hashlib.sha256("\n".join(TRAIN_KEYS).encode("utf-8")).hexdigest()[:16]

split_payload = {
    "seed": CONFIG["seed"],
    "strategy": SPLIT_STRATEGY,
    "train_signature": TRAIN_SIGNATURE,
    "train_raw": train_keys_raw,
    "train": TRAIN_KEYS,
    "val": VAL_KEYS,
    "test": TEST_KEYS,
    "exclude_zero_case_train": bool(CONFIG.get("exclude_zero_case_train", False)),
    "removed_zero_case_train": removed_zero_case_train,
    "val_selection": VAL_SELECTION_KEYS,
}
safe_json_dump(split_payload, SPLIT_PATH)

split_summary = {
    "strategy": SPLIT_STRATEGY,
    "exclude_zero_case_train": bool(CONFIG.get("exclude_zero_case_train", False)),
    "train_raw": summarize_split(train_keys_raw),
    "train": summarize_split(TRAIN_KEYS),
    "val": summarize_split(VAL_KEYS),
    "test": summarize_split(TEST_KEYS),
    "train_zero_case_removed_count": len(removed_zero_case_train),
    "train_zero_case_removed_ratio": round(len(removed_zero_case_train) / max(len(train_keys_raw), 1), 4),
    "val_selection_cases": len(VAL_SELECTION_KEYS),
    "val_selection_summary": summarize_split(VAL_SELECTION_KEYS),
}
safe_json_dump(split_summary, WORK_DIR / "logs" / "split_summary.json")

append_event(
    "split_created",
    strategy=SPLIT_STRATEGY,
    train_raw=len(train_keys_raw),
    train=len(TRAIN_KEYS),
    val=len(VAL_KEYS),
    test=len(TEST_KEYS),
    train_zero_case_removed=len(removed_zero_case_train),
    val_selection_cases=len(VAL_SELECTION_KEYS),
)

print(f"Split strategy: {SPLIT_STRATEGY}")
print(
    f"Train: {len(TRAIN_KEYS)} kept / {len(train_keys_raw)} raw | "
    f"Val: {len(VAL_KEYS)} | Test: {len(TEST_KEYS)}"
)
print("Train zero-case removed :", len(removed_zero_case_train))
print("Train zero-case kept    :", split_summary["train"]["zero_case_proxy_count"])
print("Val zero-case proxy     :", split_summary["val"]["zero_case_proxy_count"])
print("Test zero-case proxy    :", split_summary["test"]["zero_case_proxy_count"])
print(f"Validation full-volume selection subset: {len(VAL_SELECTION_KEYS)} scenarios")


Split strategy: category_and_stability
Train: 2072 kept / 2400 raw | Val: 300 | Test: 300
Train zero-case removed : 328
Train zero-case kept    : 0
Val zero-case proxy     : 49
Test zero-case proxy    : 45
Validation full-volume selection subset: 16 scenarios


In [8]:
# -- Cellule 6 : Normalization statistics --------------------------------
STATS_PATH = WORK_DIR / "normalization_stats.json"


def compute_stats(h5_path, train_keys, n_sample=300, sample_points_per_array=6000):
    rng = random.Random(CONFIG["seed"])
    sampled_keys = list(train_keys)
    rng.shuffle(sampled_keys)
    sampled_keys = sampled_keys[: min(n_sample, len(sampled_keys))]

    wind_values = {"wind_u": [], "wind_v": [], "wind_w": []}
    conc_log = []
    emiss_log = []

    with h5py.File(h5_path, "r") as handle:
        for key in tqdm(sampled_keys, desc="Compute normalization stats"):
            group = handle[key]
            for channel in ("wind_u", "wind_v", "wind_w"):
                arr = group[channel][:].ravel()
                step = max(1, len(arr) // sample_points_per_array)
                wind_values[channel].extend(arr[::step].tolist())

            conc = group["concentration_grid"][:].ravel()
            conc_step = max(1, len(conc) // sample_points_per_array)
            conc_log.extend(np.log1p(np.clip(conc[::conc_step], 0.0, None)).tolist())

            emiss = group["emissions_total"][:].ravel()
            emiss_step = max(1, len(emiss) // sample_points_per_array)
            emiss_log.extend(np.log1p(np.clip(emiss[::emiss_step], 0.0, None)).tolist())

    stats = {}
    for channel in ("wind_u", "wind_v", "wind_w"):
        values = np.asarray(wind_values[channel], dtype=np.float32)
        stats[channel] = {
            "type": "standardize",
            "mean": float(values.mean()),
            "std": max(float(values.std()), 1e-6),
        }

    stats["obstacle_sdf"] = {"type": "clip_div", "clip": 100.0}
    stats["obstacle_mask"] = {"type": "none"}
    stats["gse_sdf"] = {
        "type": "clip_div_range",
        "clip_low": -100.0,
        "clip_high": 1000.0,
        "div": 100.0,
    }
    stats["lto_sdf"] = {
        "type": "clip_div_range",
        "clip_low": -100.0,
        "clip_high": 1000.0,
        "div": 100.0,
    }

    conc_log = np.asarray(conc_log, dtype=np.float32)
    conc_positive = conc_log[conc_log > 0.01]
    conc_scale = float(np.percentile(conc_positive, 99)) if len(conc_positive) else 1.0
    stats["concentration_grid"] = {
        "type": "log1p_div",
        "scale": max(conc_scale, 0.1),
    }

    emiss_log = np.asarray(emiss_log, dtype=np.float32)
    emiss_positive = emiss_log[emiss_log > 0.01]
    emiss_scale = float(np.percentile(emiss_positive, 99)) if len(emiss_positive) else 1.0
    stats["emissions_total"] = {
        "type": "log1p_div",
        "scale": max(emiss_scale, 0.1),
    }

    return {
        "_meta": {
            "train_signature": TRAIN_SIGNATURE,
            "n_sampled_scenarios": len(sampled_keys),
            "sample_points_per_array": sample_points_per_array,
        },
        "stats": stats,
    }


reload_stats = True
if STATS_PATH.exists():
    existing_payload = json.loads(STATS_PATH.read_text(encoding="utf-8"))
    if isinstance(existing_payload, dict) and "stats" in existing_payload:
        existing_signature = existing_payload.get("_meta", {}).get("train_signature")
        if existing_signature == TRAIN_SIGNATURE:
            NORM_STATS = existing_payload["stats"]
            reload_stats = False

if reload_stats:
    stats_payload = compute_stats(CONFIG["h5_path"], TRAIN_KEYS)
    NORM_STATS = stats_payload["stats"]
    safe_json_dump(stats_payload, STATS_PATH)
    safe_json_dump(stats_payload["_meta"], WORK_DIR / "logs" / "normalization_summary.json")
    print(f"Normalization stats recomputed and saved to {STATS_PATH}")
else:
    print(f"Normalization stats loaded from {STATS_PATH}")

append_event(
    "normalization_ready",
    concentration_scale=NORM_STATS["concentration_grid"]["scale"],
    emissions_scale=NORM_STATS["emissions_total"]["scale"],
)

print("Normalization overview:")
for channel, stats in NORM_STATS.items():
    floats = {k: round(v, 4) for k, v in stats.items() if isinstance(v, float)}
    print(f"  {channel:20s} type={stats['type']:16s} {floats}")


Compute normalization stats: 100%|██████████| 300/300 [01:56<00:00,  2.57it/s]


Normalization stats recomputed and saved to nox_dispersion_run/normalization_stats.json
Normalization overview:
  wind_u               type=standardize      {'mean': 0.1801, 'std': 5.4537}
  wind_v               type=standardize      {'mean': -0.0861, 'std': 5.1638}
  wind_w               type=standardize      {'mean': -0.0001, 'std': 0.0646}
  obstacle_sdf         type=clip_div         {'clip': 100.0}
  obstacle_mask        type=none             {}
  gse_sdf              type=clip_div_range   {'clip_low': -100.0, 'clip_high': 1000.0, 'div': 100.0}
  lto_sdf              type=clip_div_range   {'clip_low': -100.0, 'clip_high': 1000.0, 'div': 100.0}
  concentration_grid   type=log1p_div        {'scale': 17.5354}
  emissions_total      type=log1p_div        {'scale': 0.1}


In [9]:
# -- Cellule 7 : Dataset -------------------------------------------------
def _norm(arr, stats):
    arr = arr.astype(np.float32)
    norm_type = stats["type"]

    if norm_type == "standardize":
        return (arr - stats["mean"]) / stats["std"]
    if norm_type == "clip_div":
        clip_value = stats["clip"]
        return np.clip(arr, -clip_value, clip_value) / clip_value
    if norm_type == "clip_div_range":
        low = stats["clip_low"]
        high = stats["clip_high"]
        div = stats["div"]
        return np.clip(arr, low, high) / div
    if norm_type == "log1p_div":
        return np.log1p(np.clip(arr, 0.0, None)) / stats["scale"]
    return arr


def denorm_concentration(arr_norm, norm_stats):
    scale = norm_stats["concentration_grid"]["scale"]
    return np.expm1(np.maximum(arr_norm * scale, 0.0))


def build_arrays_from_group(group, norm_stats, geom_mode):
    wu = _norm(group["wind_u"][:], norm_stats["wind_u"])
    wv = _norm(group["wind_v"][:], norm_stats["wind_v"])
    ww = _norm(group["wind_w"][:], norm_stats["wind_w"])
    sdf = _norm(group["obstacle_sdf"][:], norm_stats["obstacle_sdf"])
    msk = group["obstacle_mask"][:].astype(np.float32)
    gse = _norm(group["gse_sdf"][:], norm_stats["gse_sdf"])
    lto = _norm(group["lto_sdf"][:], norm_stats["lto_sdf"])
    ems = _norm(group["emissions_total"][:], norm_stats["emissions_total"])
    conc = _norm(group["concentration_grid"][:], norm_stats["concentration_grid"])

    if geom_mode == "sdf":
        geom = sdf
        x = np.stack([wu, wv, ww, sdf, gse, lto, ems]).astype(np.float32)
    elif geom_mode == "mask":
        geom = msk
        x = np.stack([wu, wv, ww, msk, gse, lto, ems]).astype(np.float32)
    else:
        geom = sdf
        x = np.stack([wu, wv, ww, sdf, msk, gse, lto, ems]).astype(np.float32)

    y = conc[None].astype(np.float32)
    geom = geom[None].astype(np.float32)
    return x, y, geom


def stable_hash_int(text):
    return int(hashlib.sha256(text.encode("utf-8")).hexdigest()[:16], 16)


def crop_or_pad(arr, top, left, crop_yx):
    if crop_yx is None:
        return arr.copy()

    pad_y = max(0, crop_yx - arr.shape[-2])
    pad_x = max(0, crop_yx - arr.shape[-1])
    if pad_y or pad_x:
        pads = [(0, 0)] * (arr.ndim - 2) + [(0, pad_y), (0, pad_x)]
        arr = np.pad(arr, pads, mode="constant")

    return arr[..., top:top + crop_yx, left:left + crop_yx].copy()


def positive_projection(volume):
    arr = np.asarray(volume, dtype=np.float32)
    if arr.ndim == 4:
        arr = arr[0]
    arr = np.maximum(arr, 0.0)
    if arr.ndim == 3:
        score = arr.max(axis=0) + 0.35 * arr.sum(axis=0)
    else:
        score = arr
    return np.nan_to_num(score, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def box_sum_2d(score_map, win_h, win_w):
    padded = np.pad(score_map, ((1, 0), (1, 0)), mode="constant")
    integral = padded.cumsum(axis=0).cumsum(axis=1)
    return (
        integral[win_h:, win_w:]
        - integral[:-win_h, win_w:]
        - integral[win_h:, :-win_w]
        + integral[:-win_h, :-win_w]
    )


def best_crop_from_score(score_map, crop_yx, deterministic, rng, top_quantile):
    if crop_yx is None:
        return 0, 0

    height, width = score_map.shape
    if crop_yx >= height and crop_yx >= width:
        return 0, 0

    win_h = min(crop_yx, height)
    win_w = min(crop_yx, width)
    window_scores = box_sum_2d(score_map, win_h, win_w)
    if window_scores.size == 0:
        return None

    flat_scores = np.nan_to_num(window_scores.ravel(), nan=0.0, posinf=0.0, neginf=0.0)
    max_score = float(np.max(flat_scores)) if flat_scores.size else 0.0
    if max_score <= 0.0:
        return None

    if deterministic:
        choice = int(flat_scores.argmax())
    else:
        threshold = float(np.quantile(flat_scores, top_quantile)) if flat_scores.size > 1 else max_score
        threshold = min(threshold, max_score)
        candidate_idx = np.flatnonzero(flat_scores >= threshold)
        if candidate_idx.size == 0:
            candidate_idx = np.array([int(flat_scores.argmax())], dtype=np.int64)
        choice = int(rng.choice(candidate_idx.tolist()))

    top, left = divmod(choice, window_scores.shape[1])
    max_top = max(0, height - crop_yx)
    max_left = max(0, width - crop_yx)

    if not deterministic:
        top = int(np.clip(top + rng.randint(-4, 4), 0, max_top))
        left = int(np.clip(left + rng.randint(-4, 4), 0, max_left))

    return int(top), int(left)


def focus_score_maps(x, y):
    target_score = positive_projection(y)
    emission_score = positive_projection(x[-1])
    return target_score, emission_score


class NOxDataset(Dataset):
    """
    Training uses hybrid plume-aware crops.
    Validation uses target-guided crops for quick monitoring. Full-volume subset evaluation
    remains label-free at inference time and is used for best-model selection.
    """

    def __init__(
        self,
        h5_path,
        keys,
        norm_stats,
        geom_mode="sdf",
        crop_yx=128,
        augment=False,
        crop_mode="auto",
        base_seed=42,
        cfg=None,
    ):
        self.h5_path = h5_path
        self.keys = list(keys)
        self.stats = norm_stats
        self.geom_mode = geom_mode
        self.crop_yx = crop_yx
        self.augment = augment
        self.crop_mode = crop_mode
        self.base_seed = base_seed
        self.cfg = dict(cfg or {})
        self._h5 = None

    def _open(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, "r")

    def __len__(self):
        return len(self.keys)

    def _resolved_crop_mode(self):
        if self.crop_mode != "auto":
            return self.crop_mode
        return "hybrid_focus" if self.augment else "full"

    def _random_offsets(self, height, width, rng):
        max_top = max(0, height - self.crop_yx)
        max_left = max(0, width - self.crop_yx)
        return rng.randint(0, max_top), rng.randint(0, max_left)

    def _crop_offsets(self, key, x, y):
        crop_mode = self._resolved_crop_mode()
        if crop_mode == "full" or self.crop_yx is None:
            return 0, 0

        _, _, height, width = x.shape
        if crop_mode == "center":
            return max(0, height - self.crop_yx) // 2, max(0, width - self.crop_yx) // 2

        seeded_rng = random.Random(stable_hash_int(f"{self.base_seed}:{key}:{crop_mode}"))
        global_rng = random

        if crop_mode == "seeded":
            return self._random_offsets(height, width, seeded_rng)
        if crop_mode == "random":
            return self._random_offsets(height, width, global_rng)

        target_score, emission_score = focus_score_maps(x, y)
        focus_quantile = float(self.cfg.get("focus_top_quantile", 0.99))

        if crop_mode == "seeded_target":
            offsets = best_crop_from_score(
                target_score,
                self.crop_yx,
                deterministic=True,
                rng=seeded_rng,
                top_quantile=focus_quantile,
            )
            return offsets if offsets is not None else self._random_offsets(height, width, seeded_rng)

        if crop_mode == "seeded_emission":
            offsets = best_crop_from_score(
                emission_score,
                self.crop_yx,
                deterministic=True,
                rng=seeded_rng,
                top_quantile=focus_quantile,
            )
            return offsets if offsets is not None else self._random_offsets(height, width, seeded_rng)

        if crop_mode == "hybrid_focus":
            target_prob = float(self.cfg.get("train_target_focus_prob", 0.85))
            emission_prob = float(self.cfg.get("train_emission_focus_prob", 0.10))
            random_prob = float(self.cfg.get("train_random_crop_prob", max(0.0, 1.0 - target_prob - emission_prob)))
            draw = global_rng.random()

            if draw < target_prob:
                offsets = best_crop_from_score(
                    target_score,
                    self.crop_yx,
                    deterministic=False,
                    rng=global_rng,
                    top_quantile=focus_quantile,
                )
                if offsets is not None:
                    return offsets

            if draw < target_prob + emission_prob:
                offsets = best_crop_from_score(
                    emission_score,
                    self.crop_yx,
                    deterministic=False,
                    rng=global_rng,
                    top_quantile=focus_quantile,
                )
                if offsets is not None:
                    return offsets

            return self._random_offsets(height, width, global_rng)

        return self._random_offsets(height, width, seeded_rng if not self.augment else global_rng)

    def __getitem__(self, idx):
        self._open()
        key = self.keys[idx]
        group = self._h5[key]
        x, y, geom = build_arrays_from_group(group, self.stats, self.geom_mode)

        crop_mode = self._resolved_crop_mode()
        if crop_mode != "full":
            top, left = self._crop_offsets(key, x, y)
            x = crop_or_pad(x, top, left, self.crop_yx)
            y = crop_or_pad(y, top, left, self.crop_yx)
            geom = crop_or_pad(geom, top, left, self.crop_yx)

        if self.augment:
            if random.random() > 0.5:
                x = x[:, :, :, ::-1].copy()
                y = y[:, :, :, ::-1].copy()
                geom = geom[:, :, :, ::-1].copy()
                x[0] *= -1.0
            if random.random() > 0.5:
                x = x[:, :, ::-1, :].copy()
                y = y[:, :, ::-1, :].copy()
                geom = geom[:, :, ::-1, :].copy()
                x[1] *= -1.0

        return (
            torch.from_numpy(x.copy()),
            torch.from_numpy(y.copy()),
            torch.from_numpy(geom.copy()),
        )

    def __del__(self):
        if self._h5 is not None:
            try:
                self._h5.close()
            except Exception:
                pass


N_IN = 8 if CONFIG["geom_mode"] == "both" else 7
print(f"Input channels: {N_IN}")
print(
    "Dataset class ready. "
    f"Train crop mode={CONFIG['train_crop_mode']} | Val/Test crop mode={CONFIG['val_crop_mode']}"
)


Input channels: 7
Dataset class ready. Train crop mode=hybrid_focus | Val/Test crop mode=seeded_target


In [10]:
# -- Cellule 8 : DataLoaders --------------------------------------------
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


loader_generator = torch.Generator()
loader_generator.manual_seed(CONFIG["seed"])

train_ds = NOxDataset(
    CONFIG["h5_path"],
    TRAIN_KEYS,
    NORM_STATS,
    geom_mode=CONFIG["geom_mode"],
    crop_yx=CONFIG["crop_yx"],
    augment=True,
    crop_mode=CONFIG["train_crop_mode"],
    base_seed=CONFIG["seed"],
    cfg=CONFIG,
)
val_ds = NOxDataset(
    CONFIG["h5_path"],
    VAL_KEYS,
    NORM_STATS,
    geom_mode=CONFIG["geom_mode"],
    crop_yx=CONFIG["crop_yx"],
    augment=False,
    crop_mode=CONFIG["val_crop_mode"],
    base_seed=CONFIG["seed"],
    cfg=CONFIG,
)
test_ds = NOxDataset(
    CONFIG["h5_path"],
    TEST_KEYS,
    NORM_STATS,
    geom_mode=CONFIG["geom_mode"],
    crop_yx=CONFIG["crop_yx"],
    augment=False,
    crop_mode=CONFIG["val_crop_mode"],
    base_seed=CONFIG["seed"],
    cfg=CONFIG,
)

num_workers = CONFIG["num_workers"]
pin_memory = DEVICE == "cuda"
base_loader_kwargs = {
    "pin_memory": pin_memory,
    "worker_init_fn": seed_worker,
    "generator": loader_generator,
}
if num_workers > 0:
    base_loader_kwargs.update(
        {
            "num_workers": num_workers,
            "persistent_workers": True,
            "prefetch_factor": 2,
        }
    )
else:
    base_loader_kwargs["num_workers"] = 0

train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    drop_last=len(train_ds) >= CONFIG["batch_size"],
    **base_loader_kwargs,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    drop_last=False,
    **base_loader_kwargs,
)
test_loader = DataLoader(
    test_ds,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory,
)

append_event(
    "dataloaders_ready",
    train_batches=len(train_loader),
    val_batches=len(val_loader),
    test_batches=len(test_loader),
    train_crop_mode=CONFIG["train_crop_mode"],
    val_crop_mode=CONFIG["val_crop_mode"],
    selection_eval_cases=len(VAL_SELECTION_KEYS),
)

print(f"Train: {len(train_ds)} scenarios | {len(train_loader)} batches/epoch")
print(f"Val  : {len(val_ds)} scenarios | {len(val_loader)} batches")
print(f"Test : {len(test_ds)} scenarios")
print(f"Validation full-volume subset: {len(VAL_SELECTION_KEYS)} scenarios")

x_s, y_s, g_s = next(iter(train_loader))
y_s_ugm3 = denorm_concentration(y_s.numpy(), NORM_STATS)
active_ratio = float((y_s_ugm3 >= CONFIG["plume_active_threshold_ugm3"]).mean())
high_ratio = float((y_s_ugm3 >= CONFIG["plume_high_threshold_ugm3"]).mean())
print(f"Sanity batch: x={tuple(x_s.shape)} y={tuple(y_s.shape)} geom={tuple(g_s.shape)}")
print(f"x range [{x_s.min():.3f}, {x_s.max():.3f}] | y range [{y_s.min():.3f}, {y_s.max():.3f}]")
print(
    f"Active plume voxels >= {CONFIG['plume_active_threshold_ugm3']:.0f} ug/m3: {active_ratio:.4%} | "
    f">= {CONFIG['plume_high_threshold_ugm3']:.0f} ug/m3: {high_ratio:.4%}"
)
del x_s, y_s, g_s, y_s_ugm3


Train: 2072 scenarios | 259 batches/epoch
Val  : 300 scenarios | 38 batches
Test : 300 scenarios
Validation full-volume subset: 16 scenarios
Sanity batch: x=(8, 7, 64, 224, 224) y=(8, 1, 64, 224, 224) geom=(8, 1, 64, 224, 224)
x range [-85.652, 154.140] | y range [0.000, 0.891]
Active plume voxels >= 10 ug/m3: 0.0296% | >= 50 ug/m3: 0.0296%


In [11]:
# -- Cellule 9 : 3D UNet -------------------------------------------------
class DoubleConv3d(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.1, inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class Down3d(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(nn.MaxPool3d(2), DoubleConv3d(in_ch, out_ch))

    def forward(self, x):
        return self.net(x)


class Up3d(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv3d(in_ch // 2 + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode="trilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


class UNet3D(nn.Module):
    def __init__(self, in_channels=7, base_ch=32):
        super().__init__()
        c = base_ch
        self.e0 = DoubleConv3d(in_channels, c)
        self.e1 = Down3d(c, c * 2)
        self.e2 = Down3d(c * 2, c * 4)
        self.e3 = Down3d(c * 4, c * 8)
        self.bn = Down3d(c * 8, c * 16)
        self.d3 = Up3d(c * 16, c * 8, c * 8)
        self.d2 = Up3d(c * 8, c * 4, c * 4)
        self.d1 = Up3d(c * 4, c * 2, c * 2)
        self.d0 = Up3d(c * 2, c, c)
        self.out = nn.Sequential(nn.Conv3d(c, 1, kernel_size=1), nn.Softplus(beta=1.0))
        self._init_weights()
        nn.init.constant_(self.out[0].bias, -3.0)

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv3d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="leaky_relu")
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.InstanceNorm3d) and module.weight is not None:
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x):
        e0 = self.e0(x)
        e1 = self.e1(e0)
        e2 = self.e2(e1)
        e3 = self.e3(e2)
        b = self.bn(e3)
        d3 = self.d3(b, e3)
        d2 = self.d2(d3, e2)
        d1 = self.d1(d2, e1)
        d0 = self.d0(d1, e0)
        return self.out(d0)

    def n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


MODEL = UNet3D(in_channels=N_IN, base_ch=CONFIG["base_ch"]).to(DEVICE)
N_PARAMS = MODEL.n_params()
MODEL_INFO = {
    "in_channels": N_IN,
    "base_ch": CONFIG["base_ch"],
    "trainable_params": int(N_PARAMS),
    "trainable_params_millions": round(N_PARAMS / 1e6, 3),
    "under_2m_budget": bool(N_PARAMS < 2_000_000),
    "notes": CONFIG["notes"],
}
safe_json_dump(MODEL_INFO, WORK_DIR / "logs" / "model_info.json")
append_event("model_built", trainable_params=int(N_PARAMS))

print(f"UNet3D: {N_PARAMS:,} trainable parameters ({N_PARAMS / 1e6:.2f}M)")
if not MODEL_INFO["under_2m_budget"]:
    print("Note: this run is accuracy-first and exceeds a strict 2M parameter budget.")

with torch.no_grad():
    probe = torch.randn(1, N_IN, 64, CONFIG["crop_yx"], CONFIG["crop_yx"], device=DEVICE)
    out = MODEL(probe)
    print(f"Forward check: {tuple(probe.shape)} -> {tuple(out.shape)}")
    del probe, out

if DEVICE == "cuda":
    torch.cuda.empty_cache()



UNet3D: 22,583,457 trainable parameters (22.58M)
Note: this run is accuracy-first and exceeds a strict 2M parameter budget.
Forward check: (1, 7, 64, 224, 224) -> (1, 1, 64, 224, 224)


In [12]:
# ── Cellule 10 : Chargement de losses.py ────────────────────────────────────

_candidates = [
    Path('provided/losses.py'),
    Path('../provided/losses.py'),
]
_losses_path = next((p for p in _candidates if p.exists()), None)

if _losses_path is None:
    raise FileNotFoundError(
        'losses.py introuvable !\n'
        'Placer provided/losses.py à côté de ce notebook (ou un niveau au-dessus).'
    )

_spec = importlib.util.spec_from_file_location('losses', str(_losses_path))
_mod  = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

dispersion_loss        = _mod.dispersion_loss
boundary_weighted_mse  = _mod.boundary_weighted_mse
mass_conservation_loss = _mod.mass_conservation_loss
tv_loss                = _mod.tv_loss

print(f'Losses chargées depuis : {_losses_path}')
print('Fonctions disponibles : dispersion_loss, boundary_weighted_mse, '
      'mass_conservation_loss, tv_loss')

Losses chargées depuis : provided/losses.py
Fonctions disponibles : dispersion_loss, boundary_weighted_mse, mass_conservation_loss, tv_loss


In [31]:
# -- Cellule 11 : Training utilities -------------------------------------
CLASS_THRESHOLDS = np.asarray(CONFIG["classification_bins"], dtype=np.float32)
CLASS_LABELS = list(CONFIG["classification_labels"])


def concentration_ugm3_to_norm(values_ugm3, norm_stats):
    scale = float(norm_stats["concentration_grid"]["scale"])
    values = np.asarray(values_ugm3, dtype=np.float32)
    return np.log1p(np.clip(values, 0.0, None)) / scale


ACTIVE_THRESHOLD_NORM = float(
    concentration_ugm3_to_norm(CONFIG["plume_active_threshold_ugm3"], NORM_STATS)
)
HIGH_THRESHOLD_NORM = float(
    concentration_ugm3_to_norm(CONFIG["plume_high_threshold_ugm3"], NORM_STATS)
)


def build_checkpoint_state(model, optimizer, scheduler, scaler, epoch, best_score, best_metric_name):
    best_score_value = None if best_score is None else float(best_score)
    return {
        "epoch": int(epoch),
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict() if scaler is not None else None,
        "best_score": best_score_value,
        "best_metric_name": str(best_metric_name),
        "best_metric_mode": str(CONFIG.get("selection_mode", "min")),
        "best_val_loss": best_score_value if str(best_metric_name) == "loss" and best_score_value is not None else None,
        "config": CONFIG,
    }


def atomic_torch_save(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp_path)
    tmp_path.replace(path)


def save_checkpoint(state, path, is_best=False):
    atomic_torch_save(state, path)
    if is_best:
        atomic_torch_save(state, Path(path).parent / "best.pt")


def load_checkpoint(path, model, cfg, optimizer=None, scheduler=None, scaler=None):
    checkpoint = torch.load(path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    if optimizer is not None and checkpoint.get("optimizer_state") is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state"])
    if scheduler is not None and checkpoint.get("scheduler_state") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state"])
    if scaler is not None and checkpoint.get("scaler_state") is not None:
        scaler.load_state_dict(checkpoint["scaler_state"])

    start_epoch = int(checkpoint.get("epoch", -1)) + 1
    best_metric_name = str(checkpoint.get("best_metric_name") or "loss")
    best_score = checkpoint.get("best_score", checkpoint.get("best_val_loss"))
    if best_score is not None:
        best_score = float(best_score)

    current_metric_name = str(cfg.get("selection_metric", "loss"))
    if best_metric_name != current_metric_name:
        print(
            f"Checkpoint best metric was '{best_metric_name}', "
            f"but current run selects on '{current_metric_name}'. Resetting best-score tracking."
        )
        best_score = None
        best_metric_name = current_metric_name

    score_text = "None" if best_score is None else f"{best_score:.6f}"
    print(f"Resume from epoch {start_epoch + 1} | best {best_metric_name}={score_text}")
    return start_epoch, best_score, best_metric_name


class Logger:
    FIELDS = [
        "epoch",
        "time_s",
        "lr",
        "gpu_peak_mem_gb",
        "train_loss",
        "train_loss_base",
        "train_mse",
        "train_mass",
        "train_solid",
        "train_tv",
        "train_loss_plume_focus",
        "train_loss_active_mae",
        "train_active_fraction",
        "train_high_fraction",
        "val_loss",
        "val_loss_base",
        "val_mse",
        "val_mass",
        "val_solid",
        "val_tv",
        "val_loss_plume_focus",
        "val_loss_active_mae",
        "val_active_fraction",
        "val_high_fraction",
        "val_rmse",
        "val_mae",
        "val_r2",
        "val_rel_err",
        "val_plume_rmse",
        "val_plume_mae",
        "val_plume_r2",
        "val_plume_rel_err",
        "val_plume_voxel_fraction",
        "val_acc",
        "val_macro_f1",
        "val_weighted_f1",
        "selection_cases",
        "selection_rmse",
        "selection_mae",
        "selection_plume_rmse",
        "selection_plume_mae",
        "selection_acc",
        "selection_macro_f1",
        "selection_weighted_f1",
        "selection_metric",
        "selection_score",
    ]

    def __init__(self, log_dir):
        self.log_dir = Path(log_dir)
        self.csv_path = self.log_dir / "training_log.csv"
        self.json_path = self.log_dir / "training_log.json"
        self.history = []
        if not self.csv_path.exists():
            with open(self.csv_path, "w", newline="", encoding="utf-8") as handle:
                csv.DictWriter(handle, fieldnames=self.FIELDS).writeheader()

    def load(self):
        if self.json_path.exists():
            self.history = json.loads(self.json_path.read_text(encoding="utf-8"))
            print(f"Loaded existing history: {len(self.history)} epochs")

    def reset(self, backup_suffix=None):
        backup_suffix = backup_suffix or time.strftime("%Y%m%d_%H%M%S")
        if self.csv_path.exists():
            self.csv_path.replace(self.log_dir / f"training_log_{backup_suffix}.csv.bak")
        if self.json_path.exists():
            self.json_path.replace(self.log_dir / f"training_log_{backup_suffix}.json.bak")
        self.history = []
        with open(self.csv_path, "w", newline="", encoding="utf-8") as handle:
            csv.DictWriter(handle, fieldnames=self.FIELDS).writeheader()

    def write(self, row):
        filtered = {field: row.get(field) for field in self.FIELDS}
        self.history.append(filtered)
        with open(self.csv_path, "a", newline="", encoding="utf-8") as handle:
            csv.DictWriter(handle, fieldnames=self.FIELDS, extrasaction="ignore").writerow(filtered)
        safe_json_dump(self.history, self.json_path)


LOGGER = Logger(WORK_DIR / "logs")


def round_or_none(value, digits=6):
    if value is None:
        return None
    value = float(value)
    if not np.isfinite(value):
        return None
    return round(value, digits)


def to_class_indices(values_ugm3):
    values = np.asarray(values_ugm3, dtype=np.float32).ravel()
    return np.digitize(values, CLASS_THRESHOLDS[1:], right=False).astype(np.int64)


def build_confusion_matrix(true_idx, pred_idx, n_classes=None):
    n_classes = n_classes or len(CLASS_LABELS)
    true_idx = np.asarray(true_idx, dtype=np.int64).ravel()
    pred_idx = np.asarray(pred_idx, dtype=np.int64).ravel()
    mask = (
        (true_idx >= 0)
        & (true_idx < n_classes)
        & (pred_idx >= 0)
        & (pred_idx < n_classes)
    )
    encoded = true_idx[mask] * n_classes + pred_idx[mask]
    return np.bincount(encoded, minlength=n_classes * n_classes).reshape(n_classes, n_classes)


def confusion_metrics(cm):
    cm = np.asarray(cm, dtype=np.int64)
    total = int(cm.sum())
    true_support = cm.sum(axis=1).astype(np.float64)
    pred_support = cm.sum(axis=0).astype(np.float64)
    tp = np.diag(cm).astype(np.float64)

    precision = tp / np.maximum(pred_support, 1.0)
    recall = tp / np.maximum(true_support, 1.0)
    f1 = 2.0 * precision * recall / np.maximum(precision + recall, 1e-12)

    return {
        "acc": float(tp.sum() / max(total, 1)),
        "macro_f1": float(f1.mean()) if len(f1) else 0.0,
        "weighted_f1": float((f1 * true_support).sum() / max(total, 1)),
        "precision_per_class": precision.tolist(),
        "recall_per_class": recall.tolist(),
        "f1_per_class": f1.tolist(),
        "support_per_class": true_support.astype(int).tolist(),
    }


def classification_metrics_from_ugm3(pred_ugm3, tgt_ugm3):
    pred_idx = to_class_indices(pred_ugm3)
    true_idx = to_class_indices(tgt_ugm3)
    cm = build_confusion_matrix(true_idx, pred_idx, len(CLASS_LABELS))
    metrics = confusion_metrics(cm)
    metrics["confusion_matrix"] = cm
    return metrics


def regression_metrics_from_ugm3(pred_ugm3, tgt_ugm3, plume_threshold_ugm3=None):
    pred = np.asarray(pred_ugm3, dtype=np.float32).ravel()
    tgt = np.asarray(tgt_ugm3, dtype=np.float32).ravel()
    plume_threshold = float(plume_threshold_ugm3 or CONFIG["plume_active_threshold_ugm3"])

    rmse = float(np.sqrt(np.mean((pred - tgt) ** 2)))
    mae = float(np.mean(np.abs(pred - tgt)))
    ss_res = float(np.sum((pred - tgt) ** 2))
    ss_tot = float(np.sum((tgt - tgt.mean()) ** 2))
    r2 = float(1.0 - ss_res / (ss_tot + 1e-10))
    rel_err = float(np.mean(np.abs(pred - tgt)) / (np.mean(np.abs(tgt)) + 1e-6))

    plume_mask = tgt >= plume_threshold
    plume_fraction = float(plume_mask.mean())
    if plume_mask.any():
        plume_pred = pred[plume_mask]
        plume_tgt = tgt[plume_mask]
        plume_ss_res = float(np.sum((plume_pred - plume_tgt) ** 2))
        plume_ss_tot = float(np.sum((plume_tgt - plume_tgt.mean()) ** 2))
        plume_rmse = float(np.sqrt(np.mean((plume_pred - plume_tgt) ** 2)))
        plume_mae = float(np.mean(np.abs(plume_pred - plume_tgt)))
        plume_r2 = float(1.0 - plume_ss_res / (plume_ss_tot + 1e-10))
        plume_rel_err = float(
            np.mean(np.abs(plume_pred - plume_tgt)) / (np.mean(np.abs(plume_tgt)) + 1e-6)
        )
    else:
        plume_rmse = None
        plume_mae = None
        plume_r2 = None
        plume_rel_err = None

    return {
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "rel_err": rel_err,
        "plume_rmse": plume_rmse,
        "plume_mae": plume_mae,
        "plume_r2": plume_r2,
        "plume_rel_err": plume_rel_err,
        "plume_voxel_fraction": plume_fraction,
    }


def regression_metrics(pred_norm, tgt_norm, norm_stats):
    pred = denorm_concentration(pred_norm, norm_stats)
    tgt = denorm_concentration(tgt_norm, norm_stats)
    return regression_metrics_from_ugm3(pred, tgt, CONFIG["plume_active_threshold_ugm3"])


def autocast_ctx(device, enabled, amp_dtype):
    if not enabled or device != "cuda":
        return nullcontext()
    dtype = torch.bfloat16 if amp_dtype == "bfloat16" else torch.float16
    return torch.autocast(device_type="cuda", dtype=dtype)


def is_scaler_enabled(cfg, device):
    return bool(device == "cuda" and cfg["use_amp"] and cfg["amp_dtype"] == "float16")


def loss_value(item):
    if torch.is_tensor(item):
        return float(item.detach().item())
    return float(item)


def plume_focus_losses(pred, target, cfg):
    pred_f = pred.float()
    target_f = target.float()
    abs_err = (pred_f - target_f).abs()
    sq_err = abs_err.pow(2)

    active_mask = target_f >= ACTIVE_THRESHOLD_NORM
    high_mask = target_f >= HIGH_THRESHOLD_NORM
    active_float = active_mask.float()
    high_float = high_mask.float()

    voxel_weight = pred_f.new_full(pred_f.shape, float(cfg.get("background_voxel_weight", 0.05)))
    voxel_weight = voxel_weight + cfg["active_voxel_weight"] * active_float + cfg["high_voxel_weight"] * high_float
    loss_plume_focus = (voxel_weight * sq_err).sum() / voxel_weight.sum().clamp(min=1.0)

    if bool(active_mask.any().item()):
        loss_active_mae = abs_err[active_mask].mean()
    else:
        loss_active_mae = abs_err.new_tensor(0.0)

    return {
        "loss_plume_focus": loss_plume_focus,
        "loss_active_mae": loss_active_mae,
        "active_fraction": active_float.mean(),
        "high_fraction": high_float.mean(),
    }


def compute_loss_dict(pred, target, geom, cfg):
    base = dispersion_loss(
        pred,
        target,
        geom,
        lambda_mass=cfg["lambda_mass"],
        lambda_tv=cfg["lambda_tv"],
        boundary_band=cfg["boundary_band"],
        boundary_weight=cfg["boundary_weight"],
    )
    plume = plume_focus_losses(pred, target, cfg)
    total = (
        base["loss"]
        + cfg["lambda_plume_focus"] * plume["loss_plume_focus"]
        + cfg["lambda_active_mae"] * plume["loss_active_mae"]
    )
    base["loss_base"] = base["loss"]
    base["loss"] = total
    base.update(plume)
    return base


def compute_selection_score(metrics, cfg):
    metric_name = str(cfg.get("selection_metric", "loss"))
    metric_value = metrics.get(metric_name)
    if metric_value is None or not np.isfinite(float(metric_value)):
        metric_name = "loss"
        metric_value = float(metrics["loss"])
    return metric_name, float(metric_value), str(cfg.get("selection_mode", "min"))


def is_better_score(current, best, mode):
    if best is None:
        return True
    if mode == "max":
        return current > best
    return current < best


def select_best_history_row(history, cfg):
    if not history:
        return None

    metric_name = str(cfg.get("selection_metric", "loss"))
    mode = str(cfg.get("selection_mode", "min"))

    def row_score(row):
        row_metric = row.get("selection_metric") or metric_name
        value = row.get("selection_score")
        if row_metric != metric_name or value is None:
            value = row.get(metric_name)
        if value is None:
            if metric_name != "loss":
                return float("-inf") if mode == "max" else float("inf")
            value = row.get("val_loss")
        return float(value)

    if mode == "max":
        return max(history, key=row_score)
    return min(history, key=row_score)


def mean_metric(rows, metric_name):
    values = [row.get(metric_name) for row in rows if row.get(metric_name) is not None]
    if not values:
        return None
    return float(np.mean(np.asarray(values, dtype=np.float64)))


@torch.no_grad()
def evaluate_selection_subset(model, keys, h5_path, norm_stats, cfg, device):
    if not keys:
        return {}

    rows = []
    cm = np.zeros((len(CLASS_LABELS), len(CLASS_LABELS)), dtype=np.int64)

    for key in tqdm(keys, desc="Val full subset", leave=False):
        pred_ugm3, tgt_ugm3, attrs, infer_meta = run_inference_for_key(
            model,
            h5_path,
            key,
            norm_stats,
            cfg,
            device,
        )
        reg_metrics = regression_metrics_from_ugm3(
            pred_ugm3,
            tgt_ugm3,
            cfg["plume_active_threshold_ugm3"],
        )
        cls_metrics = classification_metrics_from_ugm3(pred_ugm3, tgt_ugm3)
        cm += cls_metrics.pop("confusion_matrix")

        row = {
            "key": key,
            "inference_mode": infer_meta["mode"],
        }
        row.update(reg_metrics)
        row.update({
            "acc": cls_metrics["acc"],
            "macro_f1": cls_metrics["macro_f1"],
            "weighted_f1": cls_metrics["weighted_f1"],
        })
        rows.append(row)

    voxel_metrics = confusion_metrics(cm)
    return {
        "selection_cases": len(rows),
        "selection_rmse": mean_metric(rows, "rmse"),
        "selection_mae": mean_metric(rows, "mae"),
        "selection_plume_rmse": mean_metric(rows, "plume_rmse"),
        "selection_plume_mae": mean_metric(rows, "plume_mae"),
        "selection_acc": voxel_metrics["acc"],
        "selection_macro_f1": voxel_metrics["macro_f1"],
        "selection_weighted_f1": voxel_metrics["weighted_f1"],
    }


def train_one_epoch(model, loader, optimizer, scaler, cfg, device):
    model.train()
    totals = defaultdict(float)
    tracked_keys = [
        "loss",
        "loss_base",
        "loss_mse",
        "loss_mass",
        "loss_solid",
        "loss_tv",
        "loss_plume_focus",
        "loss_active_mae",
        "active_fraction",
        "high_fraction",
    ]
    scaler_enabled = is_scaler_enabled(cfg, device)
    batch_count = 0

    for x, y, geom in tqdm(loader, desc="Train", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        geom = geom.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast_ctx(device, cfg["use_amp"], cfg["amp_dtype"]):
            pred = model(x)
            loss_dict = compute_loss_dict(pred, y, geom, cfg)

        if scaler_enabled:
            scaler.scale(loss_dict["loss"]).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss_dict["loss"].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        for key in tracked_keys:
            totals[key] += loss_value(loss_dict[key])
        batch_count += 1

    return {key: totals[key] / max(batch_count, 1) for key in tracked_keys}


@torch.no_grad()
def val_one_epoch(model, loader, cfg, device, norm_stats, n_max_batches=None):
    model.eval()
    totals = defaultdict(float)
    tracked_keys = [
        "loss",
        "loss_base",
        "loss_mse",
        "loss_mass",
        "loss_solid",
        "loss_tv",
        "loss_plume_focus",
        "loss_active_mae",
        "active_fraction",
        "high_fraction",
    ]
    pred_samples = []
    tgt_samples = []
    batch_count = 0

    for batch_idx, (x, y, geom) in enumerate(tqdm(loader, desc="Val", leave=False)):
        if n_max_batches is not None and batch_idx >= n_max_batches:
            break

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        geom = geom.to(device, non_blocking=True)

        with autocast_ctx(device, cfg["use_amp"], cfg["amp_dtype"]):
            pred = model(x)
            loss_dict = compute_loss_dict(pred, y, geom, cfg)

        for key in tracked_keys:
            totals[key] += loss_value(loss_dict[key])

        pred_np = pred.detach().float().cpu().numpy()
        tgt_np = y.detach().float().cpu().numpy()
        stride = max(1, pred_np.size // cfg["val_metric_points_per_batch"])
        pred_samples.append(pred_np.ravel()[::stride])
        tgt_samples.append(tgt_np.ravel()[::stride])
        batch_count += 1

    metrics = {key: totals[key] / max(batch_count, 1) for key in tracked_keys}
    if pred_samples:
        pred_vec = np.concatenate(pred_samples)
        tgt_vec = np.concatenate(tgt_samples)
        metrics.update(regression_metrics(pred_vec, tgt_vec, norm_stats))
        cls_metrics = classification_metrics_from_ugm3(
            denorm_concentration(pred_vec, norm_stats),
            denorm_concentration(tgt_vec, norm_stats),
        )
        metrics["val_confusion_matrix"] = cls_metrics.pop("confusion_matrix")
        metrics.update(cls_metrics)

    return metrics


def blend_window(height, width):
    wy = np.hanning(height) if height > 1 else np.ones(1, dtype=np.float32)
    wx = np.hanning(width) if width > 1 else np.ones(1, dtype=np.float32)
    window = np.outer(wy, wx).astype(np.float32)
    return np.maximum(window, 1e-3)


@torch.no_grad()
def predict_volume(model, x_np, cfg, device, tile_yx=None, overlap=None):
    model.eval()
    x_np = np.asarray(x_np, dtype=np.float32)
    x_tensor = torch.from_numpy(x_np).unsqueeze(0).to(device)

    try:
        with autocast_ctx(device, cfg["use_amp"], cfg["amp_dtype"]):
            pred = model(x_tensor).detach().float().cpu().numpy()[0]
        return pred, {"mode": "direct"}
    except RuntimeError as exc:
        if "out of memory" not in str(exc).lower():
            raise
        if device == "cuda":
            torch.cuda.empty_cache()
        gc.collect()

    tile_yx = int(tile_yx or cfg.get("full_eval_tile_yx", cfg["crop_yx"]))
    overlap = int(overlap or cfg.get("full_eval_overlap", 32))
    stride = max(16, tile_yx - overlap)

    _, z_size, height, width = x_np.shape
    output = np.zeros((1, z_size, height, width), dtype=np.float32)
    weight = np.zeros((1, z_size, height, width), dtype=np.float32)
    base_window = blend_window(tile_yx, tile_yx)[None, None]

    for top in tqdm(range(0, height, stride), desc="Tiled inference", leave=False):
        for left in range(0, width, stride):
            bottom = min(top + tile_yx, height)
            right = min(left + tile_yx, width)

            patch = x_np[..., top:bottom, left:right]
            pad_y = tile_yx - patch.shape[-2]
            pad_x = tile_yx - patch.shape[-1]
            if pad_y or pad_x:
                patch = np.pad(
                    patch,
                    [(0, 0), (0, 0), (0, pad_y), (0, pad_x)],
                    mode="constant",
                )

            patch_tensor = torch.from_numpy(patch).unsqueeze(0).to(device)
            with autocast_ctx(device, cfg["use_amp"], cfg["amp_dtype"]):
                pred_patch = model(patch_tensor).detach().float().cpu().numpy()[0]

            pred_patch = pred_patch[..., : bottom - top, : right - left]
            window = base_window[..., : bottom - top, : right - left]
            output[..., top:bottom, left:right] += pred_patch * window
            weight[..., top:bottom, left:right] += window

    output /= np.maximum(weight, 1e-6)
    return output, {
        "mode": "tiled",
        "tile_yx": int(tile_yx),
        "overlap": int(overlap),
        "stride": int(stride),
    }


@torch.no_grad()
def run_inference_for_key(model, h5_path, key, norm_stats, cfg, device):
    with h5py.File(h5_path, "r") as handle:
        group = handle[key]
        attrs = {name: as_jsonable(group.attrs[name]) for name in group.attrs.keys()}
        x_np, y_np, geom_np = build_arrays_from_group(group, norm_stats, cfg["geom_mode"])

    pred_norm, infer_meta = predict_volume(model, x_np, cfg, device)
    pred_ugm3 = denorm_concentration(pred_norm, norm_stats).squeeze(0)
    tgt_ugm3 = denorm_concentration(y_np, norm_stats).squeeze(0)
    return pred_ugm3, tgt_ugm3, attrs, infer_meta


print("Training utilities ready.")
print(
    f"Plume thresholds: active>={CONFIG['plume_active_threshold_ugm3']:.0f} ug/m3, "
    f"high>={CONFIG['plume_high_threshold_ugm3']:.0f} ug/m3"
)
print(f"Full-volume validation subset: {len(VAL_SELECTION_KEYS)} scenarios")


Training utilities ready.
Plume thresholds: active>=10 ug/m3, high>=50 ug/m3
Full-volume validation subset: 16 scenarios


In [14]:
# -- Cellule 12 : Training ----------------------------------------------
CKPT_DIR = WORK_DIR / "checkpoints"
LATEST_PT = CKPT_DIR / "latest.pt"
INTERRUPTED_PT = CKPT_DIR / "interrupted_latest.pt"

optimizer = torch.optim.AdamW(
    MODEL.parameters(),
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG["n_epochs"],
    eta_min=CONFIG["lr_min"],
)
scaler = torch.amp.GradScaler(enabled=is_scaler_enabled(CONFIG, DEVICE))

start_epoch = 0
best_score = None
best_metric_name = CONFIG["selection_metric"]
LOGGER.load()

resume_candidate = None
if CONFIG["resume_training"]:
    if LATEST_PT.exists():
        resume_candidate = LATEST_PT
    elif INTERRUPTED_PT.exists():
        resume_candidate = INTERRUPTED_PT

if resume_candidate is not None:
    start_epoch, best_score, best_metric_name = load_checkpoint(
        resume_candidate,
        MODEL,
        CONFIG,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
    )
    print(f"Resuming from {resume_candidate}")
else:
    print("Starting a fresh training run.")
    if LOGGER.history:
        backup_suffix = time.strftime("%Y%m%d_%H%M%S")
        LOGGER.reset(backup_suffix=backup_suffix)
        append_event("training_history_reset", backup_suffix=backup_suffix)
        print(f"Existing history found without checkpoint; archived logs with suffix {backup_suffix}.")

print(
    f"Training for up to {CONFIG['n_epochs']} epochs | "
    f"start epoch {start_epoch + 1} | device={DEVICE}"
)
print(
    "Loss = base dispersion + "
    f"{CONFIG['lambda_plume_focus']}*plume_focus + "
    f"{CONFIG['lambda_active_mae']}*active_mae"
)
print(
    f"Model selection = {CONFIG['selection_metric']} ({CONFIG['selection_mode']}) | "
    f"train_crop={CONFIG['train_crop_mode']} | val_crop={CONFIG['val_crop_mode']}"
)
append_event(
    "training_start",
    start_epoch=start_epoch + 1,
    total_epochs=CONFIG["n_epochs"],
    selection_metric=CONFIG["selection_metric"],
    selection_mode=CONFIG["selection_mode"],
)

try:
    for epoch in range(start_epoch, CONFIG["n_epochs"]):
        if DEVICE == "cuda":
            torch.cuda.reset_peak_memory_stats()

        append_event("epoch_start", epoch=epoch + 1)
        epoch_t0 = time.time()

        train_metrics = train_one_epoch(MODEL, train_loader, optimizer, scaler, CONFIG, DEVICE)
        val_metrics = val_one_epoch(MODEL, val_loader, CONFIG, DEVICE, NORM_STATS)
        selection_metrics = evaluate_selection_subset(
            MODEL,
            VAL_SELECTION_KEYS,
            CONFIG["h5_path"],
            NORM_STATS,
            CONFIG,
            DEVICE,
        )
        combined_val_metrics = dict(val_metrics)
        combined_val_metrics.update(selection_metrics)
        scheduler.step()

        elapsed = time.time() - epoch_t0
        lr_now = float(optimizer.param_groups[0]["lr"])
        gpu_peak_mem_gb = 0.0
        if DEVICE == "cuda":
            gpu_peak_mem_gb = torch.cuda.max_memory_allocated() / 1e9

        selection_metric_name, selection_score, selection_mode = compute_selection_score(combined_val_metrics, CONFIG)
        is_best = is_better_score(selection_score, best_score, selection_mode)
        if is_best:
            best_score = float(selection_score)
            best_metric_name = selection_metric_name

        row = {
            "epoch": epoch + 1,
            "time_s": round(elapsed, 2),
            "lr": f"{lr_now:.3e}",
            "gpu_peak_mem_gb": round(gpu_peak_mem_gb, 3),
            "train_loss": round_or_none(train_metrics["loss"], 6),
            "train_loss_base": round_or_none(train_metrics["loss_base"], 6),
            "train_mse": round_or_none(train_metrics["loss_mse"], 6),
            "train_mass": round_or_none(train_metrics["loss_mass"], 6),
            "train_solid": round_or_none(train_metrics["loss_solid"], 6),
            "train_tv": round_or_none(train_metrics["loss_tv"], 6),
            "train_loss_plume_focus": round_or_none(train_metrics["loss_plume_focus"], 6),
            "train_loss_active_mae": round_or_none(train_metrics["loss_active_mae"], 6),
            "train_active_fraction": round_or_none(train_metrics["active_fraction"], 6),
            "train_high_fraction": round_or_none(train_metrics["high_fraction"], 6),
            "val_loss": round_or_none(val_metrics["loss"], 6),
            "val_loss_base": round_or_none(val_metrics["loss_base"], 6),
            "val_mse": round_or_none(val_metrics["loss_mse"], 6),
            "val_mass": round_or_none(val_metrics["loss_mass"], 6),
            "val_solid": round_or_none(val_metrics["loss_solid"], 6),
            "val_tv": round_or_none(val_metrics["loss_tv"], 6),
            "val_loss_plume_focus": round_or_none(val_metrics["loss_plume_focus"], 6),
            "val_loss_active_mae": round_or_none(val_metrics["loss_active_mae"], 6),
            "val_active_fraction": round_or_none(val_metrics["active_fraction"], 6),
            "val_high_fraction": round_or_none(val_metrics["high_fraction"], 6),
            "val_rmse": round_or_none(val_metrics.get("rmse"), 4),
            "val_mae": round_or_none(val_metrics.get("mae"), 4),
            "val_r2": round_or_none(val_metrics.get("r2"), 6),
            "val_rel_err": round_or_none(val_metrics.get("rel_err"), 6),
            "val_plume_rmse": round_or_none(val_metrics.get("plume_rmse"), 4),
            "val_plume_mae": round_or_none(val_metrics.get("plume_mae"), 4),
            "val_plume_r2": round_or_none(val_metrics.get("plume_r2"), 6),
            "val_plume_rel_err": round_or_none(val_metrics.get("plume_rel_err"), 6),
            "val_plume_voxel_fraction": round_or_none(val_metrics.get("plume_voxel_fraction"), 6),
            "val_acc": round_or_none(val_metrics.get("acc"), 6),
            "val_macro_f1": round_or_none(val_metrics.get("macro_f1"), 6),
            "val_weighted_f1": round_or_none(val_metrics.get("weighted_f1"), 6),
            "selection_cases": selection_metrics.get("selection_cases"),
            "selection_rmse": round_or_none(selection_metrics.get("selection_rmse"), 4),
            "selection_mae": round_or_none(selection_metrics.get("selection_mae"), 4),
            "selection_plume_rmse": round_or_none(selection_metrics.get("selection_plume_rmse"), 4),
            "selection_plume_mae": round_or_none(selection_metrics.get("selection_plume_mae"), 4),
            "selection_acc": round_or_none(selection_metrics.get("selection_acc"), 6),
            "selection_macro_f1": round_or_none(selection_metrics.get("selection_macro_f1"), 6),
            "selection_weighted_f1": round_or_none(selection_metrics.get("selection_weighted_f1"), 6),
            "selection_metric": selection_metric_name,
            "selection_score": round_or_none(selection_score, 6),
        }
        LOGGER.write(row)

        state = build_checkpoint_state(
            MODEL,
            optimizer,
            scheduler,
            scaler,
            epoch=epoch,
            best_score=best_score,
            best_metric_name=best_metric_name,
        )
        save_checkpoint(state, LATEST_PT, is_best=is_best)
        if (epoch + 1) % CONFIG["save_every"] == 0:
            save_checkpoint(state, CKPT_DIR / f"epoch_{epoch + 1:04d}.pt")

        append_event(
            "epoch_end",
            epoch=epoch + 1,
            is_best=is_best,
            val_loss=val_metrics["loss"],
            val_rmse=val_metrics.get("rmse", 0.0),
            val_plume_rmse=val_metrics.get("plume_rmse"),
            val_macro_f1=val_metrics.get("macro_f1", 0.0),
            selection_plume_rmse=selection_metrics.get("selection_plume_rmse"),
            selection_macro_f1=selection_metrics.get("selection_macro_f1"),
            selection_metric=selection_metric_name,
            selection_score=selection_score,
            lr=lr_now,
            time_s=elapsed,
        )

        star = " *best*" if is_best else ""
        plume_rmse_text = "n/a" if val_metrics.get("plume_rmse") is None else f"{val_metrics['plume_rmse']:.3f}"
        selection_plume_text = "n/a" if selection_metrics.get("selection_plume_rmse") is None else f"{selection_metrics['selection_plume_rmse']:.3f}"
        print(
            f"Ep {epoch + 1:4d}/{CONFIG['n_epochs']} | "
            f"tr={train_metrics['loss']:.5f} | "
            f"vl={val_metrics['loss']:.5f} | "
            f"cropPRMSE={plume_rmse_text} | "
            f"selPRMSE={selection_plume_text} | "
            f"cropF1={val_metrics.get('macro_f1', 0.0):.4f} | "
            f"selF1={selection_metrics.get('selection_macro_f1', 0.0):.4f} | "
            f"acc={val_metrics.get('acc', 0.0):.4f} | "
            f"selAcc={selection_metrics.get('selection_acc', 0.0):.4f} | "
            f"sel={selection_metric_name}:{selection_score:.4f} | "
            f"lr={lr_now:.2e} | "
            f"peak_mem={gpu_peak_mem_gb:.2f} GB{star}"
        )

    append_event(
        "training_completed",
        best_metric_name=best_metric_name,
        best_score=best_score,
        epochs=len(LOGGER.history),
    )
    print("Training finished.")
    print(f"Best {best_metric_name}: {best_score if best_score is not None else 'None'}")
    print(f"Logs: {LOGGER.csv_path}")

except KeyboardInterrupt:
    interrupted_epoch = epoch + 1 if "epoch" in locals() else start_epoch
    interrupted_state = build_checkpoint_state(
        MODEL,
        optimizer,
        scheduler,
        scaler,
        epoch=max(interrupted_epoch - 1, 0),
        best_score=best_score,
        best_metric_name=best_metric_name,
    )
    save_checkpoint(interrupted_state, INTERRUPTED_PT, is_best=False)
    append_event("training_interrupted", epoch=interrupted_epoch)
    print(f"Training interrupted. Last completed epoch is still in {LATEST_PT}.")
    print(f"An interruption snapshot was also saved to {INTERRUPTED_PT}.")
    raise

except Exception as exc:
    interrupted_epoch = epoch + 1 if "epoch" in locals() else start_epoch
    failure_state = build_checkpoint_state(
        MODEL,
        optimizer,
        scheduler,
        scaler,
        epoch=max(interrupted_epoch - 1, 0),
        best_score=best_score,
        best_metric_name=best_metric_name,
    )
    save_checkpoint(failure_state, INTERRUPTED_PT, is_best=False)
    append_event(
        "training_failed",
        epoch=interrupted_epoch,
        error=str(exc),
        traceback=traceback.format_exc(),
    )
    print(f"Training failed at epoch {interrupted_epoch}: {exc}")
    print(f"Saved interruption snapshot to {INTERRUPTED_PT}")
    raise


Starting a fresh training run.
Training for up to 200 epochs | start epoch 1 | device=cuda
Loss = base dispersion + 1.0*plume_focus + 0.15*active_mae
Model selection = selection_plume_rmse (min) | train_crop=hybrid_focus | val_crop=seeded_target


Ep    1/200 | tr=0.13603 | vl=0.02898 | cropPRMSE=727027.938 | selPRMSE=816058.982 | cropF1=0.1705 | selF1=0.1662 | acc=0.9512 | selAcc=0.9510 | sel=selection_plume_rmse:816058.9823 | lr=3.00e-04 | peak_mem=33.34 GB *best*


Ep    2/200 | tr=0.03000 | vl=0.02215 | cropPRMSE=727038.250 | selPRMSE=150694.037 | cropF1=0.1773 | selF1=0.1704 | acc=0.9696 | selAcc=0.9727 | sel=selection_plume_rmse:150694.0374 | lr=3.00e-04 | peak_mem=31.51 GB *best*


Ep    3/200 | tr=0.02426 | vl=0.02119 | cropPRMSE=727032.000 | selPRMSE=138395.081 | cropF1=0.1761 | selF1=0.1691 | acc=0.9713 | selAcc=0.9766 | sel=selection_plume_rmse:138395.0814 | lr=3.00e-04 | peak_mem=31.51 GB *best*


Ep    4/200 | tr=0.02182 | vl=0.02045 | cropPRMSE=727031.688 | selPRMSE=134886.909 | cropF1=0.1776 | selF1=0.1738 | acc=0.9730 | selAcc=0.9773 | sel=selection_plume_rmse:134886.9092 | lr=3.00e-04 | peak_mem=31.51 GB *best*


...

Ep  196/200 | tr=0.00755 | vl=0.00855 | cropPRMSE=727123.062 | selPRMSE=134237.816 | cropF1=0.2267 | selF1=0.2247 | acc=0.9960 | selAcc=0.9980 | sel=selection_plume_rmse:134237.8161 | lr=1.30e-06 | peak_mem=31.51 GB


Ep  197/200 | tr=0.00754 | vl=0.00857 | cropPRMSE=727123.375 | selPRMSE=134231.075 | cropF1=0.2268 | selF1=0.2246 | acc=0.9960 | selAcc=0.9980 | sel=selection_plume_rmse:134231.0748 | lr=1.17e-06 | peak_mem=31.51 GB


Ep  198/200 | tr=0.00758 | vl=0.00859 | cropPRMSE=727124.375 | selPRMSE=134241.278 | cropF1=0.2267 | selF1=0.2246 | acc=0.9960 | selAcc=0.9980 | sel=selection_plume_rmse:134241.2781 | lr=1.07e-06 | peak_mem=31.51 GB


Ep  199/200 | tr=0.00758 | vl=0.00852 | cropPRMSE=727120.438 | selPRMSE=134231.175 | cropF1=0.2264 | selF1=0.2243 | acc=0.9960 | selAcc=0.9980 | sel=selection_plume_rmse:134231.1748 | lr=1.02e-06 | peak_mem=31.51 GB


Ep  200/200 | tr=0.00750 | vl=0.00851 | cropPRMSE=727121.875 | selPRMSE=134234.000 | cropF1=0.2263 | selF1=0.2241 | acc=0.9960 | selAcc=0.9980 | sel=selection_plume_rmse:134233.9999 | lr=1.00e-06 | peak_mem=31.51 GB
Training finished.
Best selection_plume_rmse: 134094.38624954224
Logs: nox_dispersion_run/logs/training_log.csv


In [32]:
# ── Cellule 13 : Chargement du meilleur modèle ───────────────────────────────

BEST_PT = WORK_DIR / "checkpoints" / "best.pt"
if BEST_PT.exists():
    ckpt = torch.load(BEST_PT, map_location=DEVICE)
    MODEL.load_state_dict(ckpt["model_state"])
    MODEL.eval()
    ep = ckpt.get("epoch", "?")
    best_metric_name = ckpt.get("best_metric_name", "loss")
    best_score = ckpt.get("best_score", ckpt.get("best_val_loss", "?"))
    if isinstance(best_score, (int, float)):
        score_text = f"{best_score:.6f}"
    else:
        score_text = str(best_score)
    print(f"Meilleur modele charge (epoque {ep}, {best_metric_name}={score_text})")
else:
    print("Aucun best.pt trouve - chargement de latest.pt")
    ckpt = torch.load(WORK_DIR / "checkpoints" / "latest.pt", map_location=DEVICE)
    MODEL.load_state_dict(ckpt["model_state"])
    MODEL.eval()
    print(f"latest.pt charge (epoque {ckpt.get('epoch', '?')})")

LOGGER.load()
print(f"Historique : {len(LOGGER.history)} epoques")


Meilleur modele charge (epoque 162, selection_plume_rmse=134094.386250)
Loaded existing history: 200 epochs
Historique : 200 epoques


In [33]:
# -- Cellule 14 : Full test evaluation ----------------------------------
EVAL_PATH = WORK_DIR / "logs" / "test_evaluation.json"
TEST_SUMMARY_PATH = WORK_DIR / "logs" / "test_summary.json"
TEST_CM_PATH = WORK_DIR / "logs" / "test_confusion_matrix.json"
REPORT_SUMMARY_PATH = WORK_DIR / "logs" / "report_summary.json"


def fmt_metric(value, digits=3):
    if value is None:
        return "n/a"
    return f"{float(value):.{digits}f}"


def summarize_metric_group(rows):
    if not rows:
        return {}

    summary = {"n": len(rows)}
    metric_names = [
        "rmse",
        "mae",
        "r2",
        "rel_err",
        "plume_rmse",
        "plume_mae",
        "plume_r2",
        "plume_rel_err",
        "plume_voxel_fraction",
        "acc",
        "macro_f1",
        "weighted_f1",
    ]
    for name in metric_names:
        values = [row.get(name) for row in rows if row.get(name) is not None]
        if values:
            values = np.asarray(values, dtype=np.float64)
            summary[f"{name}_mean"] = float(values.mean())
            summary[f"{name}_std"] = float(values.std())
        else:
            summary[f"{name}_mean"] = None
            summary[f"{name}_std"] = None

    summary["zero_case_proxy_count"] = int(sum(int(row["zero_case_proxy"]) for row in rows))
    return summary


@torch.no_grad()
def full_test_eval(model, test_keys, h5_path, norm_stats, cfg, device):
    model.eval()
    results = []
    test_cm = np.zeros((len(CLASS_LABELS), len(CLASS_LABELS)), dtype=np.int64)
    inference_modes = Counter()

    append_event("test_eval_start", n_test=len(test_keys))

    for key in tqdm(test_keys, desc="Full test evaluation"):
        pred_ugm3, gt_ugm3, attrs, infer_meta = run_inference_for_key(
            model,
            h5_path,
            key,
            norm_stats,
            cfg,
            device,
        )
        reg_metrics = regression_metrics_from_ugm3(
            pred_ugm3,
            gt_ugm3,
            cfg["plume_active_threshold_ugm3"],
        )
        cls_metrics = classification_metrics_from_ugm3(pred_ugm3, gt_ugm3)
        test_cm += cls_metrics.pop("confusion_matrix")
        inference_modes[infer_meta["mode"]] += 1

        result = {
            "key": key,
            "shape": list(gt_ugm3.shape),
            "category": int(attrs.get("category", -1)),
            "stability": str(attrs.get("stability_class", "?")),
            "U_10": float(attrs.get("U_10", 0.0)),
            "n_active": int(attrs.get("n_active_particles", 0)),
            "Nx": int(attrs.get("Nx", gt_ugm3.shape[-1])),
            "zero_case_proxy": int(attrs.get("n_active_particles", 0)) == 0,
            "inference_mode": infer_meta["mode"],
            "tile_yx": infer_meta.get("tile_yx"),
            "overlap": infer_meta.get("overlap"),
        }
        result.update(reg_metrics)
        result.update({
            "acc": cls_metrics["acc"],
            "macro_f1": cls_metrics["macro_f1"],
            "weighted_f1": cls_metrics["weighted_f1"],
        })
        results.append(result)

    scenario_overall = summarize_metric_group(results)
    voxel_classification = confusion_metrics(test_cm)
    summary = {
        "scenario_overall": scenario_overall,
        "voxel_classification": voxel_classification,
        "by_category": {
            str(cat): summarize_metric_group([row for row in results if row["category"] == cat])
            for cat in sorted({row["category"] for row in results})
        },
        "by_stability": {
            stab: summarize_metric_group([row for row in results if row["stability"] == stab])
            for stab in sorted({row["stability"] for row in results})
        },
        "by_zero_case_proxy": {
            "false": summarize_metric_group([row for row in results if not row["zero_case_proxy"]]),
            "true": summarize_metric_group([row for row in results if row["zero_case_proxy"]]),
        },
        "inference_modes": dict(inference_modes),
        "class_labels": CLASS_LABELS,
        "selection_metric": cfg["selection_metric"],
    }

    safe_json_dump(results, EVAL_PATH)
    safe_json_dump(summary, TEST_SUMMARY_PATH)
    safe_json_dump({"labels": CLASS_LABELS, "matrix": test_cm.tolist()}, TEST_CM_PATH)

    best_epoch_row = select_best_history_row(LOGGER.history, CONFIG)
    report_summary = {
        "best_epoch": best_epoch_row,
        "selection_metric": cfg["selection_metric"],
        "scenario_overall": scenario_overall,
        "voxel_classification": voxel_classification,
        "model_info": MODEL_INFO,
    }
    safe_json_dump(report_summary, REPORT_SUMMARY_PATH)

    append_event(
        "test_eval_completed",
        n_test=len(results),
        rmse_mean=scenario_overall.get("rmse_mean"),
        plume_rmse_mean=scenario_overall.get("plume_rmse_mean"),
        macro_f1_mean=scenario_overall.get("macro_f1_mean"),
    )

    return results, summary, test_cm


TEST_RESULTS, TEST_SUMMARY, TEST_CM = full_test_eval(
    MODEL,
    TEST_KEYS,
    CONFIG["h5_path"],
    NORM_STATS,
    CONFIG,
    DEVICE,
)

print("=" * 72)
print(f"Full test evaluation completed on {len(TEST_RESULTS)} scenarios")
print("=" * 72)
print(f"Scenario RMSE mean      : {fmt_metric(TEST_SUMMARY['scenario_overall'].get('rmse_mean'))}")
print(f"Scenario plume RMSE mean: {fmt_metric(TEST_SUMMARY['scenario_overall'].get('plume_rmse_mean'))}")
print(f"Scenario MAE mean       : {fmt_metric(TEST_SUMMARY['scenario_overall'].get('mae_mean'))}")
print(f"Scenario R2 mean        : {fmt_metric(TEST_SUMMARY['scenario_overall'].get('r2_mean'), 4)}")
print(f"Scenario Macro-F1       : {fmt_metric(TEST_SUMMARY['scenario_overall'].get('macro_f1_mean'), 4)}")
print(f"Scenario Accuracy       : {fmt_metric(TEST_SUMMARY['scenario_overall'].get('acc_mean'), 4)}")
print("Inference modes         :", TEST_SUMMARY["inference_modes"])
print(f"Saved per-scenario results to {EVAL_PATH}")
print(f"Saved summary to {TEST_SUMMARY_PATH}")


Full test evaluation: 100%|██████████| 300/300 [05:15<00:00,  1.05s/it]

Full test evaluation completed on 300 scenarios
Scenario RMSE mean      : 7216999986.066
Scenario plume RMSE mean: 151560.592
Scenario MAE mean       : 3858911.137
Scenario R2 mean        : -2480333255773189672696924726079835340800.0000
Scenario Macro-F1       : 0.2127
Scenario Accuracy       : 0.9984
Inference modes         : {'direct': 300}
Saved per-scenario results to nox_dispersion_run/logs/test_evaluation.json
Saved summary to nox_dispersion_run/logs/test_summary.json


In [34]:
# ── Cellule 15 : Courbes d'entraînement ──────────────────────────────────────

def plot_training_curves(history, save_path):
    if not history:
        print("Pas d'historique disponible.")
        return
    epochs = [r["epoch"] for r in history]

    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    fig.suptitle("NOx Dispersion - Historique d'entrainement", fontsize=14)

    def _plot(ax, keys_labels, title, yscale="linear"):
        for key, label, color in keys_labels:
            values = [r.get(key, None) for r in history]
            values = [v for v in values if v is not None]
            ax.plot(epochs[:len(values)], values, label=label, color=color, lw=1.5)
        ax.set_title(title)
        ax.set_xlabel("Epoque")
        ax.set_yscale(yscale)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    _plot(
        axes[0, 0],
        [("train_loss", "Train total", "steelblue"), ("val_loss", "Val total", "tomato")],
        "Loss totale",
        "log",
    )

    _plot(
        axes[0, 1],
        [
            ("train_mse", "MSE", "#2196F3"),
            ("train_mass", "Mass", "#4CAF50"),
            ("train_solid", "Solid", "#FF9800"),
            ("train_tv", "TV", "#9C27B0"),
        ],
        "Composantes base train",
        "log",
    )

    _plot(
        axes[0, 2],
        [
            ("train_loss_plume_focus", "Plume focus train", "#6A1B9A"),
            ("val_loss_plume_focus", "Plume focus val", "#AD1457"),
            ("train_loss_active_mae", "Active MAE train", "#00897B"),
            ("val_loss_active_mae", "Active MAE val", "#43A047"),
        ],
        "Composantes plume",
        "log",
    )

    _plot(
        axes[0, 3],
        [
            ("selection_plume_rmse", "Selection plume RMSE", "darkred"),
            ("val_plume_rmse", "Crop plume RMSE", "firebrick"),
            ("selection_rmse", "Selection RMSE", "darkorange"),
            ("val_rmse", "Crop RMSE", "goldenrod"),
        ],
        "Validation RMSE",
    )

    _plot(
        axes[1, 0],
        [
            ("val_macro_f1", "Crop Macro-F1", "seagreen"),
            ("selection_macro_f1", "Selection Macro-F1", "darkgreen"),
            ("val_acc", "Crop Accuracy", "slateblue"),
            ("selection_acc", "Selection Accuracy", "navy"),
        ],
        "Classification validation",
    )

    _plot(
        axes[1, 1],
        [
            ("val_plume_mae", "Plume MAE", "crimson"),
            ("val_mae", "MAE global", "firebrick"),
        ],
        "Validation MAE",
    )

    _plot(
        axes[1, 2],
        [
            ("val_active_fraction", "Frac plume active", "#00796B"),
            ("val_high_fraction", "Frac plume haute", "#C62828"),
            ("val_plume_voxel_fraction", "Frac plume samplee", "#1565C0"),
        ],
        "Occupation des plumes",
    )

    lr_vals = []
    for row in history:
        try:
            lr_vals.append(float(row.get("lr", 0)))
        except (ValueError, TypeError):
            lr_vals.append(0.0)
    axes[1, 3].plot(epochs, lr_vals, color="purple", lw=1.5, label="Learning rate")
    axes[1, 3].set_title("Learning rate")
    axes[1, 3].set_yscale("log")
    axes[1, 3].set_xlabel("Epoque")
    axes[1, 3].grid(alpha=0.3)
    axes[1, 3].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Courbes sauvegardees -> {save_path}")


plot_training_curves(
    LOGGER.history,
    WORK_DIR / "visualizations" / "training_curves.png",
)


Courbes sauvegardees -> nox_dispersion_run/visualizations/training_curves.png


In [35]:
# -- Cellule 16 : Confusion matrix --------------------------------------
if "TEST_CM" not in globals():
    stored_cm = json.loads(TEST_CM_PATH.read_text(encoding="utf-8"))
    TEST_CM = np.asarray(stored_cm["matrix"], dtype=np.int64)

cm_metrics = confusion_metrics(TEST_CM)
print("Per-class concentration metrics:")
print(f"{'Class':12s}  Precision  Recall    F1      Support")
for idx, label in enumerate(CLASS_LABELS):
    print(
        f"{label:12s}  "
        f"{cm_metrics['precision_per_class'][idx]:.4f}     "
        f"{cm_metrics['recall_per_class'][idx]:.4f}   "
        f"{cm_metrics['f1_per_class'][idx]:.4f}   "
        f"{cm_metrics['support_per_class'][idx]}"
    )
print(f"\nVoxel accuracy      : {cm_metrics['acc']:.4f}")
print(f"Voxel macro F1      : {cm_metrics['macro_f1']:.4f}")
print(f"Voxel weighted F1   : {cm_metrics['weighted_f1']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("NOx concentration confusion matrix", fontsize=13)

sns.heatmap(
    TEST_CM,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_LABELS,
    yticklabels=CLASS_LABELS,
    ax=axes[0],
)
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")
axes[0].set_title("Absolute counts")

norm_cm = TEST_CM.astype(np.float64) / np.maximum(TEST_CM.sum(axis=1, keepdims=True), 1.0) * 100.0
sns.heatmap(
    norm_cm,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    xticklabels=CLASS_LABELS,
    yticklabels=CLASS_LABELS,
    ax=axes[1],
    vmin=0,
    vmax=100,
)
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")
axes[1].set_title("Recall by class (%)")

plt.tight_layout()
conf_mat_path = WORK_DIR / "visualizations" / "confusion_matrix.png"
plt.savefig(conf_mat_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved confusion matrix figure to {conf_mat_path}")


Per-class concentration metrics:
Class         Precision  Recall    F1      Support
0-10          0.9999     0.9983   0.9991   1819539910
10-50         0.0000     0.0169   0.0001   1890
50-100        0.0002     0.0096   0.0004   4694
100-200       0.0001     0.0059   0.0002   4257
200-500       0.0065     0.0722   0.0120   34648
>500          0.1606     0.5768   0.2512   574601

Voxel accuracy      : 0.9982
Voxel macro F1      : 0.2105
Voxel weighted F1   : 0.9988
Saved confusion matrix figure to nox_dispersion_run/visualizations/confusion_matrix.png


In [36]:
# -- Cellule 17 : 2D concentration slices -------------------------------
@torch.no_grad()
def plot_concentration_slices(model, h5_path, key, norm_stats, cfg, device, z_levels=(3, 10, 20, 40)):
    pred_m, gt_m, attrs, infer_meta = run_inference_for_key(
        model,
        h5_path,
        key,
        norm_stats,
        cfg,
        device,
    )
    err = pred_m - gt_m

    z_levels = [min(z, pred_m.shape[0] - 1) for z in z_levels]
    n_cols = len(z_levels)
    fig, axes = plt.subplots(3, n_cols, figsize=(5 * n_cols, 12))
    if n_cols == 1:
        axes = axes[:, None]

    combined_values = np.concatenate([gt_m.reshape(-1), pred_m.reshape(-1)])
    positive_values = combined_values[combined_values > 0]
    vmax = float(np.percentile(positive_values, 99.8)) if positive_values.size else 1.0
    vmax = max(vmax, 1.0)
    title = (
        f"{key} | cat={attrs.get('category', '?')} | U10={float(attrs.get('U_10', 0.0)):.1f} m/s | "
        f"stab={attrs.get('stability_class', '?')} | Nx={attrs.get('Nx', '?')} | mode={infer_meta['mode']}"
    )
    fig.suptitle(title, fontsize=11)

    for col, z in enumerate(z_levels):
        kw = dict(origin="lower", aspect="auto", interpolation="bilinear")

        im0 = axes[0, col].imshow(gt_m[z], cmap="hot_r", vmin=0, vmax=vmax, **kw)
        axes[0, col].set_title(f"Ground truth z={z}")
        plt.colorbar(im0, ax=axes[0, col], label="ug/m3", fraction=0.04)

        im1 = axes[1, col].imshow(pred_m[z], cmap="hot_r", vmin=0, vmax=vmax, **kw)
        axes[1, col].set_title(f"Prediction z={z}")
        plt.colorbar(im1, ax=axes[1, col], label="ug/m3", fraction=0.04)

        err_lim = max(float(np.abs(err[z]).max()), 1.0)
        im2 = axes[2, col].imshow(err[z], cmap="RdBu_r", vmin=-err_lim, vmax=err_lim, **kw)
        axes[2, col].set_title(f"Error z={z}")
        plt.colorbar(im2, ax=axes[2, col], label="ug/m3", fraction=0.04)

        for row in range(3):
            axes[row, col].axis("off")

    for label, row in zip(["Ground truth", "Prediction", "Error (pred-gt)"], range(3)):
        axes[row, 0].set_ylabel(label, fontsize=11)

    plt.tight_layout()
    save_path = WORK_DIR / "visualizations" / f"slices_{key}.png"
    plt.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"Saved slices to {save_path}")
    return pred_m, gt_m


category_examples = {}
for key in TEST_KEYS:
    category = ALL_META[key]["category"]
    if category not in category_examples:
        category_examples[category] = key
    if len(category_examples) >= 4:
        break

for category, key in sorted(category_examples.items()):
    print(f"\n--- Category {category}: {key} ---")
    plot_concentration_slices(MODEL, CONFIG["h5_path"], key, NORM_STATS, CONFIG, DEVICE)



--- Category 1: scenario_00004 ---
Saved slices to nox_dispersion_run/visualizations/slices_scenario_00004.png

--- Category 2: scenario_00009 ---
Saved slices to nox_dispersion_run/visualizations/slices_scenario_00009.png

--- Category 3: scenario_00106 ---
Saved slices to nox_dispersion_run/visualizations/slices_scenario_00106.png

--- Category 4: scenario_00103 ---
Saved slices to nox_dispersion_run/visualizations/slices_scenario_00103.png


In [37]:
# ── Cellule 18 : Scatter Prédit vs Réel + Distribution des erreurs ────────────

@torch.no_grad()
def scatter_analysis(model, loader, norm_stats, cfg, device,
                     n_batches=50, title='Val Set'):
    scale = norm_stats['concentration_grid']['scale']
    def denorm(a):
        return np.expm1(np.maximum(a * scale, 0.0))

    all_p, all_t = [], []
    model.eval()
    for i, (x, y, geom) in enumerate(loader):
        if i >= n_batches:
            break
        with torch.autocast(device_type=device, enabled=cfg['use_amp']):
            pred = model(x.to(device)).float().cpu().numpy()
        pf = pred.ravel()
        tf = y.numpy().ravel()
        step = max(1, len(pf) // 3000)
        all_p.append(denorm(pf[::step]))
        all_t.append(denorm(tf[::step]))

    p = np.concatenate(all_p).clip(0)
    t = np.concatenate(all_t).clip(0)
    errors = p - t

    # Métriques
    rmse = np.sqrt(np.mean(errors**2))
    r2   = 1 - np.sum(errors**2) / (np.sum((t - t.mean())**2) + 1e-10)
    bias = errors.mean()

    # Cap pour affichage
    cap  = np.percentile(t, 98)
    mask = t < cap

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    fig.suptitle(f'{title} — Prédiction vs Vérité terrain', fontsize=13)

    # Scatter
    ax = axes[0]
    ax.scatter(t[mask], p[mask], alpha=0.03, s=1, c='steelblue', rasterized=True)
    ax.plot([0, cap], [0, cap], 'r--', lw=1.5, label='y = x')
    ax.set_xlim(0, cap)
    ax.set_ylim(0, cap)
    ax.set_xlabel('Concentration réelle (µg/m³)')
    ax.set_ylabel('Concentration prédite (µg/m³)')
    ax.set_title(f'R²={r2:.3f}  RMSE={rmse:.1f} µg/m³')
    ax.legend()
    ax.grid(alpha=0.3)

    # Histogramme des erreurs
    ax = axes[1]
    ax.hist(errors, bins=150, color='steelblue', alpha=0.75, edgecolor='none',
            range=(np.percentile(errors, 1), np.percentile(errors, 99)))
    ax.axvline(0,    color='red',    lw=1.5, ls='--', label='0')
    ax.axvline(bias, color='orange', lw=1.5, ls='-',  label=f'biais={bias:.1f}')
    ax.set_xlabel('Erreur (µg/m³)')
    ax.set_ylabel('Fréquence')
    ax.set_title('Distribution des erreurs')
    ax.legend()
    ax.grid(alpha=0.3)

    # Erreur relative vs concentration réelle
    ax = axes[2]
    t_log = np.log1p(t[mask])
    e_abs = np.abs(errors[mask])
    bins  = np.linspace(0, np.log1p(cap), 25)
    bwidth = bins[1] - bins[0]
    bx, by, by_std = [], [], []
    for b in bins[:-1]:
        m2 = (t_log >= b) & (t_log < b + bwidth)
        if m2.sum() > 50:
            bx.append(np.expm1(b + bwidth / 2))
            by.append(e_abs[m2].mean())
            by_std.append(e_abs[m2].std())
    if bx:
        ax.plot(bx, by, 'o-', color='darkorange', lw=2, ms=4)
        ax.fill_between(bx,
                        [a - s for a, s in zip(by, by_std)],
                        [a + s for a, s in zip(by, by_std)],
                        alpha=0.3, color='darkorange')
    ax.set_xlabel('Concentration réelle (µg/m³)')
    ax.set_ylabel('MAE (µg/m³)')
    ax.set_title('Erreur absolue moyenne par niveau')
    ax.set_xscale('symlog', linthresh=1)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    p_save = WORK_DIR / 'visualizations' / 'scatter_pred_vs_gt.png'
    plt.savefig(p_save, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'RMSE={rmse:.2f} µg/m³  R²={r2:.4f}  biais={bias:.2f} µg/m³')
    print(f'Sauvegardé → {p_save}')


scatter_analysis(MODEL, val_loader, NORM_STATS, CONFIG, DEVICE, title='Val Set')

/tmp/ipykernel_4338/1673350798.py:43: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax.set_xlim(0, cap)
/tmp/ipykernel_4338/1673350798.py:44: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  ax.set_ylim(0, cap)


RMSE=35498.55 µg/m³  R²=0.0001  biais=-172.26 µg/m³
Sauvegardé → nox_dispersion_run/visualizations/scatter_pred_vs_gt.png


In [38]:
# -- Cellule 19 : Interactive 3D isosurface ------------------------------
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    from IPython.display import HTML, display

    @torch.no_grad()
    def plot_3d_isosurface(model, h5_path, key, norm_stats, cfg, device, thr_pct=0.8, ds_step=2):
        pred_m, gt_m, attrs, infer_meta = run_inference_for_key(
            model,
            h5_path,
            key,
            norm_stats,
            cfg,
            device,
        )

        pred_ds = pred_m[::ds_step, ::ds_step, ::ds_step]
        gt_ds = gt_m[::ds_step, ::ds_step, ::ds_step]
        z_size, y_size, x_size = pred_ds.shape
        z_ax = np.arange(z_size) * ds_step * 3.0
        y_ax = np.arange(y_size) * ds_step * 5.0
        x_ax = np.arange(x_size) * ds_step * 5.0
        z_grid, y_grid, x_grid = np.meshgrid(z_ax, y_ax, x_ax, indexing="ij")

        nz_vals = pred_ds[pred_ds > 1.0]
        threshold = float(np.percentile(nz_vals, (1.0 - thr_pct) * 100.0)) if len(nz_vals) else 1.0

        fig = make_subplots(
            rows=1,
            cols=2,
            specs=[[{"type": "volume"}, {"type": "volume"}]],
            subplot_titles=["Prediction", "Ground truth"],
        )

        for col, (data, label) in enumerate([(pred_ds, "Pred"), (gt_ds, "GT")], start=1):
            fig.add_trace(
                go.Volume(
                    x=x_grid.ravel(),
                    y=y_grid.ravel(),
                    z=z_grid.ravel(),
                    value=data.ravel(),
                    isomin=threshold,
                    isomax=float(data.max()),
                    opacity=0.12,
                    surface_count=12,
                    colorscale="Hot",
                    showscale=(col == 1),
                    name=label,
                ),
                row=1,
                col=col,
            )

        fig.update_layout(
            title_text=(
                f"{key} | cat={attrs.get('category', '?')} | U10={float(attrs.get('U_10', 0.0)):.1f} m/s | "
                f"mode={infer_meta['mode']} | threshold>={threshold:.1f} ug/m3"
            ),
            height=650,
        )

        html_path = WORK_DIR / "visualizations" / f"3d_{key}.html"
        fig.write_html(str(html_path))
        print(f"Saved interactive 3D plot to {html_path}")
        try:
            display(HTML(html_path.read_text(encoding="utf-8")))
        except Exception as exc:
            print(f"Inline HTML display unavailable: {exc}")
            try:
                fig.show()
            except Exception as show_exc:
                print(f"Plotly inline rendering unavailable: {show_exc}")
                print("Open the saved HTML file manually in a browser if needed.")

    def select_showcase_key(test_results, fallback_keys):
        if test_results:
            candidates = [
                row for row in test_results
                if (not row.get("zero_case_proxy")) and row.get("plume_rmse") is not None
            ]
            if candidates:
                candidates = sorted(
                    candidates,
                    key=lambda row: (
                        float(row.get("r2", -1e18)),
                        float(row.get("macro_f1", -1.0)),
                        -float(row.get("rmse", 1e18)),
                    ),
                    reverse=True,
                )
                return candidates[0]["key"], "best_nonzero_test_case"

            candidates = [row for row in test_results if not row.get("zero_case_proxy")]
            if candidates:
                candidates = sorted(
                    candidates,
                    key=lambda row: (
                        float(row.get("macro_f1", -1.0)),
                        -float(row.get("rmse", 1e18)),
                    ),
                    reverse=True,
                )
                return candidates[0]["key"], "best_nonzero_fallback"

        if fallback_keys:
            return fallback_keys[0], "first_test_key"
        return None, "no_available_key"

    showcase_key, showcase_reason = select_showcase_key(globals().get("TEST_RESULTS"), TEST_KEYS)
    if showcase_key is not None:
        print(f"Using showcase scenario for 3D view: {showcase_key} ({showcase_reason})")
        plot_3d_isosurface(MODEL, CONFIG["h5_path"], showcase_key, NORM_STATS, CONFIG, DEVICE)

except ImportError:
    print("Plotly is not available. Install it with: pip install plotly")



Using showcase scenario for 3D view: scenario_05877 (best_nonzero_test_case)
Saved interactive 3D plot to nox_dispersion_run/visualizations/3d_scenario_05877.html


In [39]:
# -- Cellule 20 : Vertical flythrough video ------------------------------
try:
    import imageio.v2 as iio

    @torch.no_grad()
    def make_z_flythrough_video(model, h5_path, key, norm_stats, cfg, device, fps=12):
        pred_m, gt_m, attrs, infer_meta = run_inference_for_key(
            model,
            h5_path,
            key,
            norm_stats,
            cfg,
            device,
        )

        z_size = pred_m.shape[0]
        combined_values = np.concatenate([gt_m.reshape(-1), pred_m.reshape(-1)])
        positive_values = combined_values[combined_values > 0]
        vmax = float(np.percentile(positive_values, 99.8)) if positive_values.size else 1.0
        vmax = max(vmax, 1.0)
        frames = []

        for z in range(z_size):
            fig, axes = plt.subplots(1, 3, figsize=(14.08, 4.0), dpi=100)
            fig.suptitle(
                f"{key} | z={z} ({z * 3} m agl) | mode={infer_meta['mode']}",
                fontsize=10,
                y=1.01,
            )
            kw = dict(origin="lower", aspect="auto", interpolation="bilinear")

            axes[0].imshow(gt_m[z], cmap="hot_r", vmin=0, vmax=vmax, **kw)
            axes[0].set_title("Ground truth")
            axes[0].axis("off")

            axes[1].imshow(pred_m[z], cmap="hot_r", vmin=0, vmax=vmax, **kw)
            axes[1].set_title("Prediction")
            axes[1].axis("off")

            err = pred_m[z] - gt_m[z]
            err_lim = max(float(np.abs(err).max()), 1.0)
            axes[2].imshow(err, cmap="RdBu_r", vmin=-err_lim, vmax=err_lim, **kw)
            axes[2].set_title("Error")
            axes[2].axis("off")

            fig.canvas.draw()
            frame_rgba = np.asarray(fig.canvas.buffer_rgba(), dtype=np.uint8)
            frames.append(frame_rgba[..., :3].copy())
            plt.close(fig)

        video_path = WORK_DIR / "visualizations" / f"video_z_{key}.mp4"
        with iio.get_writer(
            str(video_path),
            fps=fps,
            codec="libx264",
            output_params=["-crf", "23", "-pix_fmt", "yuv420p"],
        ) as writer:
            for frame in frames:
                writer.append_data(frame)

        print(f"Saved flythrough video to {video_path}")
        return video_path

    if "select_showcase_key" in globals():
        showcase_key, showcase_reason = select_showcase_key(globals().get("TEST_RESULTS"), TEST_KEYS)
    elif TEST_KEYS:
        showcase_key, showcase_reason = TEST_KEYS[0], "first_test_key"
    else:
        showcase_key, showcase_reason = None, "no_available_key"

    if showcase_key is not None:
        print(f"Using showcase scenario for video: {showcase_key} ({showcase_reason})")
        make_z_flythrough_video(MODEL, CONFIG["h5_path"], showcase_key, NORM_STATS, CONFIG, DEVICE)

except ImportError as exc:
    print(f"Video generation dependencies missing: {exc}")
    print("Install imageio[ffmpeg] and ffmpeg if needed.")
except Exception as exc:
    print(f"Video generation failed: {exc}")
    print("The notebook saved no video because frame extraction failed before encoding.")

Using showcase scenario for video: scenario_05877 (best_nonzero_test_case)


Multiple -pix_fmt options specified for stream 0, only the last option '-pix_fmt yuv420p' will be used.


Saved flythrough video to nox_dispersion_run/visualizations/video_z_scenario_05877.mp4


In [40]:
# -- Cellule 21 : Best / worst predictions ------------------------------
if "TEST_RESULTS" in globals() and TEST_RESULTS:
    by_r2 = sorted(TEST_RESULTS, key=lambda row: row["r2"], reverse=True)
    BEST_3 = by_r2[:3]
    WORST_3 = by_r2[-3:]

    print("Top 3 by R2:")
    for row in BEST_3:
        print(
            f"  {row['key']} | R2={row['r2']:.4f} | RMSE={row['rmse']:.2f} | "
            f"cat={row['category']} | U10={row['U_10']:.1f} | mode={row['inference_mode']}"
        )

    print("Bottom 3 by R2:")
    for row in WORST_3:
        print(
            f"  {row['key']} | R2={row['r2']:.4f} | RMSE={row['rmse']:.2f} | "
            f"cat={row['category']} | U10={row['U_10']:.1f} | mode={row['inference_mode']}"
        )

    def get_pred_pair(key):
        pred_m, gt_m, attrs, infer_meta = run_inference_for_key(
            MODEL,
            CONFIG["h5_path"],
            key,
            NORM_STATS,
            CONFIG,
            DEVICE,
        )
        return pred_m, gt_m, attrs, infer_meta

    rows = BEST_3 + WORST_3
    labels = ["BEST #1", "BEST #2", "BEST #3", "WORST #1", "WORST #2", "WORST #3"]

    fig = plt.figure(figsize=(22, 18))
    grid = gridspec.GridSpec(6, 4, hspace=0.4, wspace=0.3)
    fig.suptitle("Best and worst predictions (z=15)", fontsize=13)

    for row_idx, (row, label) in enumerate(zip(rows, labels)):
        pred_m, gt_m, attrs, infer_meta = get_pred_pair(row["key"])
        z = min(15, pred_m.shape[0] - 1)
        combined_slice = np.concatenate([gt_m[z].ravel(), pred_m[z].ravel()])
        positive_slice = combined_slice[combined_slice > 0]
        vmax = float(np.percentile(positive_slice, 99.8)) if positive_slice.size else 1.0
        vmax = max(vmax, 1.0)

        ax0 = fig.add_subplot(grid[row_idx, 0])
        ax0.imshow(gt_m[z], cmap="hot_r", vmin=0, vmax=vmax, origin="lower", aspect="auto")
        ax0.set_title(f"{label} GT | {row['key'][:18]}", fontsize=8)
        ax0.axis("off")

        ax1 = fig.add_subplot(grid[row_idx, 1])
        ax1.imshow(pred_m[z], cmap="hot_r", vmin=0, vmax=vmax, origin="lower", aspect="auto")
        ax1.set_title(f"Pred | R2={row['r2']:.3f}", fontsize=8)
        ax1.axis("off")

        err = pred_m[z] - gt_m[z]
        err_lim = max(float(np.abs(err).max()), 1.0)
        ax2 = fig.add_subplot(grid[row_idx, 2])
        ax2.imshow(err, cmap="RdBu_r", vmin=-err_lim, vmax=err_lim, origin="lower", aspect="auto")
        ax2.set_title(f"Err | RMSE={row['rmse']:.1f}", fontsize=8)
        ax2.axis("off")

        cy, cx = pred_m.shape[1] // 2, pred_m.shape[2] // 2
        zz = np.arange(pred_m.shape[0]) * 3
        ax3 = fig.add_subplot(grid[row_idx, 3])
        ax3.plot(gt_m[:, cy, cx], zz, label="GT", lw=2)
        ax3.plot(pred_m[:, cy, cx], zz, label="Pred", lw=2, ls="--")
        ax3.set_xlabel("ug/m3", fontsize=7)
        ax3.set_ylabel("m agl", fontsize=7)
        ax3.set_title(f"Vertical profile | {infer_meta['mode']}", fontsize=8)
        ax3.legend(fontsize=7)
        ax3.grid(alpha=0.3)

    save_path = WORK_DIR / "visualizations" / "best_worst_predictions.png"
    plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.show()
    print(f"Saved figure to {save_path}")
else:
    print("TEST_RESULTS is not available. Run the full test evaluation cell first.")



Top 3 by R2:
  scenario_05877 | R2=0.5280 | RMSE=13.43 | cat=2 | U10=1.4 | mode=direct
  scenario_09553 | R2=0.5117 | RMSE=13.02 | cat=2 | U10=5.0 | mode=direct
  scenario_09253 | R2=0.4811 | RMSE=7.39 | cat=2 | U10=7.8 | mode=direct
Bottom 3 by R2:
  scenario_02926 | R2=-929409104420536320000000000.0000 | RMSE=76215.53 | cat=3 | U10=7.7 | mode=direct
  scenario_04414 | R2=-11792531987543992331969495040000000000.0000 | RMSE=8585063936.00 | cat=3 | U10=4.1 | mode=direct
  scenario_07978 | R2=-744088184199968460567113703096320000000000.0000 | RMSE=2156513722368.00 | cat=3 | U10=1.1 | mode=direct
Saved figure to nox_dispersion_run/visualizations/best_worst_predictions.png


In [41]:
# ── Cellule 22 : Analyse par paramètres météo ────────────────────────────────

if 'TEST_RESULTS' in dir() and TEST_RESULTS:
    import matplotlib.ticker as ticker

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    fig.suptitle('Performance du modèle selon les conditions météo / scénario', fontsize=13)

    # R² vs vitesse de vent
    ax = axes[0]
    u10  = [r['U_10']  for r in TEST_RESULTS]
    r2s  = [r['r2']    for r in TEST_RESULTS]
    rmse = [r['rmse']  for r in TEST_RESULTS]
    cats = [r['category'] for r in TEST_RESULTS]
    sc   = ax.scatter(u10, r2s, c=cats, cmap='tab10', s=40, alpha=0.7)
    ax.set_xlabel('U_10 (m/s)')
    ax.set_ylabel('R²')
    ax.set_title('R² vs Vitesse de vent')
    plt.colorbar(sc, ax=ax, label='Catégorie')
    ax.grid(alpha=0.3)

    # RMSE vs catégorie
    ax = axes[1]
    cat_list = sorted(set(cats))
    box_data = [[r['rmse'] for r in TEST_RESULTS if r['category'] == c]
                for c in cat_list]
    bp = ax.boxplot(box_data, labels=[f'Cat {c}' for c in cat_list],
                    patch_artist=True, notch=True)
    colors = plt.cm.tab10(np.linspace(0, 1, len(cat_list)))
    for patch, col in zip(bp['boxes'], colors):
        patch.set_facecolor(col)
        patch.set_alpha(0.7)
    ax.set_ylabel('RMSE (µg/m³)')
    ax.set_title('RMSE par catégorie d\'aéroport')
    ax.grid(alpha=0.3)

    # R² vs n_active_particles
    ax = axes[2]
    nact = [r['n_active'] for r in TEST_RESULTS]
    ax.scatter(nact, r2s, c=rmse, cmap='hot_r', s=40, alpha=0.7)
    ax.set_xlabel('n_active_particles')
    ax.set_ylabel('R²')
    ax.set_title('R² vs Activité des particules')
    ax.set_xscale('symlog', linthresh=100)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    save_path = WORK_DIR / 'visualizations' / 'meteo_analysis.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Sauvegardé → {save_path}')
else:
    print('TEST_RESULTS non disponible.')

/tmp/ipykernel_4338/2849968039.py:27: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(box_data, labels=[f'Cat {c}' for c in cat_list],


Sauvegardé → nox_dispersion_run/visualizations/meteo_analysis.png


In [42]:
# -- Cellule 23 : Final ZIP export --------------------------------------
zip_name = f"nox_results_{time.strftime('%Y%m%d_%H%M')}.zip"
zip_path = WORK_DIR.parent / zip_name

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for folder_name in ["logs", "visualizations", "outputs"]:
        folder = WORK_DIR / folder_name
        if folder.exists():
            for file_path in folder.rglob("*"):
                if file_path.is_file():
                    archive.write(file_path, file_path.relative_to(WORK_DIR))

    for checkpoint_name in ["best.pt", "latest.pt", "interrupted_latest.pt"]:
        checkpoint_path = WORK_DIR / "checkpoints" / checkpoint_name
        if checkpoint_path.exists():
            archive.write(checkpoint_path, f"checkpoints/{checkpoint_name}")

    for metadata_name in [
        "config.json",
        "normalization_stats.json",
        "data_split.json",
    ]:
        metadata_path = WORK_DIR / metadata_name
        if metadata_path.exists():
            archive.write(metadata_path, metadata_name)

size_mb = zip_path.stat().st_size / 1e6
print(f"Exported results to {zip_path}")
print(f"ZIP size: {size_mb:.1f} MB")
print("Included key artifacts:")
print("  logs/training_log.csv and logs/training_log.json")
print("  logs/events.jsonl")
print("  logs/system_info.json")
print("  logs/dataset_inspection.json and logs/scenario_metadata.json")
print("  logs/split_summary.json")
print("  logs/model_info.json")
print("  logs/test_evaluation.json, logs/test_summary.json, logs/test_confusion_matrix.json")
print("  logs/report_summary.json")
print("  visualizations/*.png / *.html / *.mp4 when generated")
print("  checkpoints/best.pt, latest.pt, interrupted_latest.pt when present")



Exported results to nox_results_20260406_1703.zip
ZIP size: 503.5 MB
Included key artifacts:
  logs/training_log.csv and logs/training_log.json
  logs/events.jsonl
  logs/system_info.json
  logs/dataset_inspection.json and logs/scenario_metadata.json
  logs/split_summary.json
  logs/model_info.json
  logs/test_evaluation.json, logs/test_summary.json, logs/test_confusion_matrix.json
  logs/report_summary.json
  visualizations/*.png / *.html / *.mp4 when generated
  checkpoints/best.pt, latest.pt, interrupted_latest.pt when present
